# Derivatives Matching Engine — PySpark / Databricks Production

**Document Reference:** Derivatives.docx (Business Requirements Document)

## Purpose
Production-grade PySpark implementation of the Derivatives Trade Matching Engine.
Replicates **exactly the same rules and logic** as the Pandas POC but optimised for:
- **4 M Axis × 20 M Finstore** rows
- **100+ columns** per dataset
- Databricks Runtime 16.4 LTS cluster (122 GB, 16 cores, 2–10 workers)

## Architecture (Best Practices Applied)
| Principle | Implementation |
|---|---|
| Match narrow, enrich wide | Core schemas (~15 cols) for matching; wide tables joined at the end |
| Candidates → Score → Resolve | All 15 BRD rules generate candidate edges; resolved via window ranking |
| No iterative pool removal | Priority + window ranking replaces repeated anti-joins |
| Block aggressively | Counterparty blocking (Strategy 1), amount-bucket blocking (Strategy 2) |
| Delta everywhere | CSV only as final export; optimizeWrite + autoCompact enabled |
| No Python UDFs | All derivations use native Spark SQL functions |
| Minimal .count() calls | ~6 essential counts vs ~20+ in original; arithmetic derivation where possible |
| Cluster-tuned config | 320 shuffle partitions, 256 MB broadcast, AQE + skew join |
| MEMORY_AND_DISK persist | Safe spill for large DataFrames at 4M+ row scale |

## Output Tables
| Table | Schema | Description |
|---|---|---|
| `matched_all_base` | Narrow (~18 cols) | Core match metadata + 1:1 traceability (saved BEFORE enrichment) |
| `matched_all_enriched` | Wide (100+ cols) | Full enriched matches |
| `matched_brd_layer` | Narrow | BRD deterministic matches only |
| `matched_greedy_layer` | Narrow | Greedy probabilistic matches only |
| `unmatched_axis_base` | Narrow | Unmatched Axis core |
| `unmatched_axis_enriched` | Wide | Unmatched Axis full |
| `unmatched_finstore_base` | Narrow | Unmatched Finstore core |
| `unmatched_finstore_enriched` | Wide | Unmatched Finstore full |

## Layers
- **Layer 1:** BRD Deterministic Rules (15 total: 3 SOPHIS + 10 OTC + 2 ETD)
- **Layer 2:** Greedy / Probabilistic Matching
  - Strategy 1: Amount + Counterparty (1% tolerance)
  - Strategy 2: Amount-only strict (0.1% tolerance)

## Scope Note
SOPHIS and DELTA1 systems are excluded by default.
Their trade IDs do not exist in Finstore — this is a data mapping issue, not a matching algorithm issue.

**Date:** 2026-03-11 (v5.5 — 1:1 traceability columns: match_rank, is_best_match, match_type)

---
## 🔹 Section 1: Spark Session & Configuration

In [ ]:
# ============================================================
# SPARK SESSION & IMPORTS
# ============================================================
# On Databricks the SparkSession is pre-created as `spark`.
# For local testing uncomment the builder block below.

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark import StorageLevel
from functools import reduce
from datetime import datetime
import logging

# --- Uncomment for local / non-Databricks execution ---
# spark = (
#     SparkSession.builder
#     .master("local[*]")
#     .appName("DerivativesMatchingEngine")
#     .config("spark.sql.adaptive.enabled", "true")
#     .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
#     .config("spark.sql.adaptive.skewJoin.enabled", "true")
#     .config("spark.sql.shuffle.partitions", "200")
#     .config("spark.driver.memory", "8g")
#     .getOrCreate()
# )

# ============================================================
# CLUSTER-TUNED SETTINGS — 122 GB, 16 cores, 2-10 workers
# Databricks Runtime 16.4 LTS
# ============================================================

# --- AQE (Adaptive Query Execution) ---
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")

# Shuffle partitions: ~2-3× total cores across cluster
# With 10 workers × 16 cores = 160 cores → 320 partitions is a good start.
# AQE will coalesce down if too many are empty.
spark.conf.set("spark.sql.shuffle.partitions", "320")

# Broadcast threshold: 256 MB (Axis core ~4M × 11 cols ≈ 200-400 MB,
# if it fits, Spark will auto-broadcast the smaller side of BRD joins)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(256 * 1024 * 1024))

# Optimize large shuffles — tungsten sort-based shuffle for better memory use
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m")

# Reduce small file problem on Delta writes
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

print(f"Spark version: {spark.version}")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Skew join: {spark.conf.get('spark.sql.adaptive.skewJoin.enabled')}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Broadcast threshold: {int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))/(1024*1024):.0f} MB")
print(f"Delta optimizeWrite: {spark.conf.get('spark.databricks.delta.optimizeWrite.enabled')}")
print(f"Delta autoCompact: {spark.conf.get('spark.databricks.delta.autoCompact.enabled')}")

In [ ]:
# ============================================================
# SCOPE CONFIGURATION
# ============================================================

# Set to True to exclude SOPHIS and DELTA1 systems from scope
# These systems' trade IDs don't exist in Finstore (data mapping issue)
EXCLUDE_SOPHIS_DELTA1 = True

# Out-of-scope systems (when EXCLUDE_SOPHIS_DELTA1 = True)
OUT_OF_SCOPE_SYSTEMS = [
    "SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO", "SOPHISFX-LONDON",
    "DELTA1-LONDON", "DELTA1-NEWYORK"
]

# Greedy matching parameters
GREEDY_TOLERANCE_PCT = 0.01     # 1% for Strategy 1 (Amount + Counterparty)
STRICT_TOLERANCE_PCT = 0.001    # 0.1% for Strategy 2 (Amount only)
BUCKET_SIZE = 1000              # Amount bucket size for Strategy 2 blocking
MIN_GREEDY_AMOUNT = 100.0       # Minimum |amount| for greedy eligibility — prevents near-zero false positives
S1_MAX_ITER = 8                 # Strategy 1 iteration cap (typically converges in 3-5)
S2_MAX_ITER = 5                 # Strategy 2 iteration cap (broader join → fewer iters to limit cost)
LINEAGE_CHECKPOINT_EVERY = 3    # Truncate Spark DAG lineage every N iterations

# ============================================================
# GREEDY — SAP ACCOUNT FILTER  (v5.6)
# ============================================================
# Restrict the Finstore pool entering greedy to only records whose
# SAPAccountId is in a known list.  This reduces the 15M+ Finstore
# pool to a manageable subset and dramatically speeds up greedy joins.
#
# Set GREEDY_SAP_FILTER_ENABLED = False to disable (use full pool).
GREEDY_SAP_FILTER_ENABLED = True
GREEDY_SAP_FILTER_COL = "SAPAccountId"  # <-- column name in Finstore

# Known SAP Account IDs to include in greedy matching (sample list)
SAP_ACCOUNT_LIST = [
    "SAP001", "SAP002", "SAP003", "SAP004", "SAP005",
    "SAP006", "SAP007", "SAP008", "SAP009", "SAP010",
    # Add your real SAP account IDs here
]

# ============================================================
# CHECKPOINT & RESILIENCE  (v5.4)
# ============================================================
# Save BRD results before greedy starts so they survive greedy failures.
SAVE_BRD_CHECKPOINT = True      # Save brd_matches + unmatched pools to Delta after BRD
RUN_GREEDY = True               # Set False to skip greedy entirely (BRD-only run)
BRD_CHECKPOINT_DIR = f"{INPUT_DIR}/matching_results/_brd_checkpoint"

# ============================================================
# LINEAGE & AUDITABILITY  (Best Practice §6)
# ============================================================
# Every match row will carry these for full traceability.
import uuid

RUN_ID = str(uuid.uuid4())                           # unique per execution
BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")  # human-readable batch
RUN_TIMESTAMP = datetime.now().isoformat()
RULE_VERSION = "v5.5-pyspark"                         # v5.5: 1:1 traceability (match_rank, is_best_match, match_type)

# ============================================================
# FILE PATHS — Adjust for Databricks / ADLS / DBFS
# ============================================================

# For Databricks: use dbfs:/ or abfss:// paths
# For local: use file:///path/to/files
INPUT_DIR = "/mnt/data/derivatives"           # <-- adjust to your mount / path
INPUT_FILE_AXIS = f"{INPUT_DIR}/axis_sample_poc.csv"
INPUT_FILE_FINSTORE = f"{INPUT_DIR}/finstore_sample_poc.csv"
SDS_MAPPING_FILE = f"{INPUT_DIR}/sds_book_mapping.csv"

OUTPUT_DIR = f"{INPUT_DIR}/matching_results"

print("SCOPE CONFIGURATION:")
print(f"  EXCLUDE_SOPHIS_DELTA1: {EXCLUDE_SOPHIS_DELTA1}")
if EXCLUDE_SOPHIS_DELTA1:
    print(f"  Excluded systems: {', '.join(OUT_OF_SCOPE_SYSTEMS)}")
print(f"\nGREEDY MATCHING PARAMETERS:")
print(f"  Strategy 1 tolerance: {GREEDY_TOLERANCE_PCT * 100}%")
print(f"  Strategy 2 tolerance: {STRICT_TOLERANCE_PCT * 100}%")
print(f"  Bucket size: {BUCKET_SIZE}")
print(f"  S1 max iterations: {S1_MAX_ITER}")
print(f"  S2 max iterations: {S2_MAX_ITER}")
print(f"  RUN_GREEDY: {RUN_GREEDY}")
print(f"  SAP filter: {GREEDY_SAP_FILTER_ENABLED}  (col={GREEDY_SAP_FILTER_COL}, {len(SAP_ACCOUNT_LIST)} accounts)")
print(f"\nCHECKPOINT:")
print(f"  SAVE_BRD_CHECKPOINT: {SAVE_BRD_CHECKPOINT}")
print(f"  BRD_CHECKPOINT_DIR: {BRD_CHECKPOINT_DIR}")
print(f"\nLINEAGE:")
print(f"  run_id:       {RUN_ID}")
print(f"  batch_id:     {BATCH_ID}")
print(f"  rule_version: {RULE_VERSION}")
print(f"\nPATHS:")
print(f"  Input:  {INPUT_DIR}")
print(f"  Output: {OUTPUT_DIR}")

---
## 🔹 Section 2: BRD Constants & System Classifications

In [ ]:
# ============================================================
# BRD SECTION: System Classification Lists
# ============================================================

# OTC Systems (from BRD table)
OTC_SYSTEMS = [
    "ALD-SF", "BARXFX-PD-ASIAPAC", "BARXFX-PD-LONDON", "BARXFETS",
    "BARXFX-TS-CASH-ASIAPAC", "BARXFX-TS-CASH-LONDON", "DELTA1-LONDON",
    "DELTA1-NEWYORK", "IFLOW-AMERICAS", "IFLOW-ASIAPACIFIC", "IFLOW-EUROPE",
    "FISSFX-NEWYORK", "FISS-LONDON", "FISS-TOKYO", "GCD-NEWYORK",
    "IMPACT-NEWYORK", "OPENLINK-LONDON", "OPTISRD-MEXICO", "SABS-NEWYORK",
    "SOPHISFX-LONDON", "SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO",
    "SUMMIT-LONDON", "SYNFINY-CFD-NONCASH", "SYETR-NEWYORK-MBS"
]

# ETD Systems (from BRD table)
ETD_SYSTEMS = [
    "BPS222-OPT-NEWYORK", "BPS224-OPT-NEWYORK", "ODH-DOLPHIN-INDIA",
    "ODH-CMI-CPM-LONDON", "ODH-GMI-HONGKONG", "ODH-GMI-LONDON",
    "ODH-GMI-LONDON-BDT", "ODH-GMI-NEWYORK", "ODH-GMI-SINGAPORE",
    "ODH-ISTAR-TOKYO", "OPTISFUT-MEXICO"
]

# SOPHIS Systems
SOPHIS_SYSTEMS = [
    "SOPHISFX-LONDON", "SOPHIS-LONDON", "SOPHIS-TOKYO", "SOPHIS-NEWYORK"
]

# DELTA1 Systems
DELTA1_SYSTEMS = ["DELTA1-LONDON", "DELTA1-NEWYORK"]

# Amount column names
AXIS_AMOUNT_COL = "SACCRMTM"
FIN_AMOUNT_COL = "gbpequivalentamount"

print("BRD CONSTANTS LOADED")
print(f"  OTC Systems:    {len(OTC_SYSTEMS)}")
print(f"  ETD Systems:    {len(ETD_SYSTEMS)}")
print(f"  SOPHIS Systems: {len(SOPHIS_SYSTEMS)}")
print(f"  DELTA1 Systems: {len(DELTA1_SYSTEMS)}")

---
## 🔹 Section 3: Matching Rule Definitions (All 15 BRD Rules)

Rules are defined as lightweight dictionaries instead of dataclasses.
Each rule specifies:
- `priority`: global waterfall order (1 = highest priority)
- `category`: SOPHIS / OTC / ETD
- `brd_priority`: priority within category
- `axis_keys` / `fin_keys`: column(s) to join on
- `filter_sophis_only`: whether to restrict Axis to SOPHIS systems
- `requires_derived_masterbook`: skipped if DerivedMasterbookId is empty

In [ ]:
# ============================================================
# ALL 15 BRD MATCHING RULES  (v5.0 — with safe_rule_join metadata)
# ============================================================
# Each rule specifies:
#   - priority, category, brd_priority: waterfall order
#   - axis_keys / fin_keys: primary equi-join columns
#   - filter_sophis_only: restrict Axis to SOPHIS systems
#   - requires_derived_masterbook: skip if DerivedMasterbookId empty
#
# NEW optional keys for safe_rule_join:
#   - system_filter: list of systems to restrict Axis to (e.g., ETD only)
#   - extra_axis_keys / extra_fin_keys: additional equi-join predicates
#     applied as post-join filters to narrow M:M without compound keys
#   - top_k: max fin matches per axis for this rule (default: 20)

MATCHING_RULES = [
    # ---- SOPHIS-specific rules (Priority 1-3) ----
    dict(
        priority=1, category="SOPHIS", brd_priority=1,
        description="DerivedSophisId <> fissnumber",
        axis_keys=["DerivedSophisId"],
        fin_keys=["fissnumber"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=2, category="SOPHIS", brd_priority=2,
        description="DerivedSophisId + BookId <> fissnumber + tradingsystembook",
        axis_keys=["DerivedSophisId", "BookId"],
        fin_keys=["fissnumber", "tradingsystembook"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=3, category="SOPHIS", brd_priority=3,
        description="DerivedSophisId <> tradeid",
        axis_keys=["DerivedSophisId"],
        fin_keys=["tradeid"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),

    # ---- OTC rules (Priority 4-13) ----
    dict(
        priority=4, category="OTC", brd_priority=1,
        description="SourceSystemTradeId <> tradeid",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=5, category="OTC", brd_priority=2,
        description="SourceSystemTradeId + DerivedMasterbookId <> tradeid + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["tradeid", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),
    dict(
        priority=6, category="OTC", brd_priority=3,
        description="SourceSystemTradeId <> alternatetradeid1",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["alternatetradeid1"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=7, category="OTC", brd_priority=4,
        description="SourceSystemTradeId + DerivedMasterbookId <> alternatetradeid1 + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["alternatetradeid1", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),
    dict(
        priority=8, category="OTC", brd_priority=5,
        description="DerivedSophisId <> fissnumber",
        axis_keys=["DerivedSophisId"],
        fin_keys=["fissnumber"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=9, category="OTC", brd_priority=6,
        description="DerivedSophisId + BookId <> fissnumber + tradingsystembook",
        axis_keys=["DerivedSophisId", "BookId"],
        fin_keys=["fissnumber", "tradingsystembook"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=10, category="OTC", brd_priority=7,
        description="DerivedSophisId <> tradeid",
        axis_keys=["DerivedSophisId"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=11, category="OTC", brd_priority=8,
        description="DerivedDelta1Id <> tradeid",
        axis_keys=["DerivedDelta1Id"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=12, category="OTC", brd_priority=9,
        description="SourceSystemTradeId <> alternatetradeid2",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["alternatetradeid2"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=13, category="OTC", brd_priority=10,
        description="SourceSystemTradeId + DerivedMasterbookId <> alternatetradeid2 + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["alternatetradeid2", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),

    # ---- ETD rules (Priority 14-15) ----
    # P14: 2-key ETD (instrument + masterbook) — skipped when SDS unavailable
    dict(
        priority=14, category="ETD", brd_priority=1,
        description="SourceSystemInstrumentId + DerivedMasterbookId <> instrumentid + masterbookid",
        axis_keys=["SourceSystemInstrumentId", "DerivedMasterbookId"],
        fin_keys=["instrumentid", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
        system_filter=ETD_SYSTEMS,   # ← restrict to ETD systems only
        top_k=10,                     # ← 2-key rule, allow multi-leg but bounded
    ),
    # P15: 1-key ETD FALLBACK — THE KNOWN OFFENDER
    # TIGHTENED (v5.0):
    #   1. system_filter: only ETD systems (was: all systems)
    #   2. extra equi-join on SourceSystemName to narrow M:M
    #   3. top_k=5: strict cap — ETD instruments shouldn't match
    #      across many unrelated trades
    dict(
        priority=15, category="ETD", brd_priority=2,
        description="SourceSystemInstrumentId <> instrumentid [ETD-only, system-constrained]",
        axis_keys=["SourceSystemInstrumentId"],
        fin_keys=["instrumentid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
        system_filter=ETD_SYSTEMS,              # ← ONLY ETD systems participate
        extra_axis_keys=["SourceSystemName"],   # ← additional equi-join:
        extra_fin_keys=["sourcesystemname"],    #   same system on both sides
        top_k=5,                                 # ← strict multi-leg cap
    ),
]

sophis_count = sum(1 for r in MATCHING_RULES if r["category"] == "SOPHIS")
otc_count = sum(1 for r in MATCHING_RULES if r["category"] == "OTC")
etd_count = sum(1 for r in MATCHING_RULES if r["category"] == "ETD")

print(f"SOPHIS rules: {sophis_count}")
print(f"OTC rules:    {otc_count}")
print(f"ETD rules:    {etd_count}")
print(f"TOTAL:        {len(MATCHING_RULES)} rules")

# Report rules with special constraints
for r in MATCHING_RULES:
    extras = []
    if r.get("system_filter"):
        extras.append(f"system_filter={len(r['system_filter'])} systems")
    if r.get("extra_axis_keys"):
        extras.append(f"extra_join={r['extra_axis_keys']}")
    if r.get("top_k"):
        extras.append(f"top_k={r['top_k']}")
    if extras:
        print(f"  P{r['priority']:2d}: {', '.join(extras)}")

---
## 🔹 Section 4: Load Data (Bronze Layer)

In [ ]:
# ============================================================
# BRONZE LAYER — Raw Ingestion
# ============================================================
# For production: read from Delta tables instead of CSV.
# Example: spark.read.format("delta").load("dbfs:/mnt/...")
#
# ⚡ PERF: No .count() here — counts are deferred to cache materialisation
#    in Section 7 to avoid triggering 3 extra Spark jobs at load time.

logger.info("Loading Axis data...")
df_axis_full = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(INPUT_FILE_AXIS)
)
# Bronze metadata (Best Practice §2A — append-only lineage)
df_axis_full = (
    df_axis_full
    .withColumn("_ingest_timestamp", F.lit(RUN_TIMESTAMP))
    .withColumn("_source_file", F.lit(INPUT_FILE_AXIS))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

logger.info("Loading Finstore data...")
df_finstore_full = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(INPUT_FILE_FINSTORE)
)
# Bronze metadata
df_finstore_full = (
    df_finstore_full
    .withColumn("_ingest_timestamp", F.lit(RUN_TIMESTAMP))
    .withColumn("_source_file", F.lit(INPUT_FILE_FINSTORE))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

# Load SDS mapping if available (small table — good broadcast candidate)
try:
    df_sds_mapping = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(SDS_MAPPING_FILE)
    )
    sds_available = True
    logger.info("SDS mapping loaded")
except Exception:
    df_sds_mapping = None
    sds_available = False
    logger.info("SDS mapping not available — DerivedMasterbookId will be empty")

print(f"\n{'='*80}")
print("DATA LOADED (BRONZE) — counts deferred to cache materialisation")
print(f"{'='*80}")
print(f"Axis columns:    {len(df_axis_full.columns)}")
print(f"Finstore columns:{len(df_finstore_full.columns)}")
print(f"SDS Mapping:     {'Available' if sds_available else 'NOT AVAILABLE'}")

---
## 🔹 Section 5: Scope Exclusion (SOPHIS / DELTA1)

In [ ]:
# ============================================================
# SCOPE EXCLUSION
# ============================================================

print(f"\n{'='*80}")
print("SCOPE EXCLUSION")
print(f"{'='*80}")

if EXCLUDE_SOPHIS_DELTA1:
    df_axis_excluded = df_axis_full.filter(
        F.col("SourceSystemName").isin(OUT_OF_SCOPE_SYSTEMS)
    )
    df_axis = df_axis_full.filter(
        ~F.col("SourceSystemName").isin(OUT_OF_SCOPE_SYSTEMS)
    )

    # ── Exclusion breakdown by system ────────────────────────────────────
    # One Spark job to show exactly how many trades each system contributes.
    print("\n*** SOPHIS/DELTA1 SCOPE EXCLUSION ACTIVE ***\n")
    print("Excluded systems breakdown:")
    excluded_breakdown = (
        df_axis_excluded
        .groupBy("SourceSystemName")
        .count()
        .orderBy(F.col("count").desc())
        .collect()
    )
    for row in excluded_breakdown:
        print(f"  {row['SourceSystemName']:<25s} : {row['count']:>10,}")

    total_excluded = sum(row['count'] for row in excluded_breakdown)
    total_full = df_axis_full.count()
    print(f"  {'─'*39}")
    print(f"  {'TOTAL EXCLUDED':<25s} : {total_excluded:>10,}")
    print(f"  {'REMAINING IN-SCOPE':<25s} : {total_full - total_excluded:>10,}")
    print(f"  {'ORIGINAL (full)':<25s} : {total_full:>10,}")
    print(f"  Excluded %: {total_excluded/total_full*100:.1f}%")
else:
    df_axis = df_axis_full
    df_axis_excluded = spark.createDataFrame([], df_axis_full.schema)
    print("All systems in scope.")

---
## 🔹 Section 6: Pre-Reconciliation Derivations (Silver Layer)

All derivations use **native Spark SQL functions** — no Python UDFs.
- `DerivedSophisId`: 3rd part of hyphen-split `SourceSystemTradeId` (SOPHIS systems)
- `DerivedDelta1Id`: 3rd part of hyphen-split `SourceSystemTradeId` (DELTA1 systems)
- `ReconSubProduct`: ETD / OTC / OTC-Default classification
- `DerivedMasterbookId`: from SDS mapping (if available)

In [ ]:
# ============================================================
# SILVER LAYER — Derivations & Key Preparation
# ============================================================
# All done with native Spark SQL — no Python UDFs.
# ⚡ PERF: No .count() calls — derivations are lazy transformations.

# --- Stable surrogate IDs (monotonically_increasing_id is unique per run) ---
df_axis = df_axis.withColumn("axis_id", F.monotonically_increasing_id())
df_finstore = df_finstore_full.withColumn("fin_id", F.monotonically_increasing_id())

# --- Normalise Finstore system column name ─────────────────────────────
# Production Finstore has "bookedsystem" (not "sourcesystemname").
# Rename to "sourcesystemname" so all downstream code (P15 extra equi-join,
# Greedy S1, Greedy S2) can reference a single consistent column name.
if "bookedsystem" in df_finstore.columns and "sourcesystemname" not in df_finstore.columns:
    df_finstore = df_finstore.withColumnRenamed("bookedsystem", "sourcesystemname")
    print("Finstore: renamed 'bookedsystem' → 'sourcesystemname'")
elif "sourcesystemname" in df_finstore.columns:
    print("Finstore: 'sourcesystemname' already present")
else:
    # Neither column exists — create an empty placeholder so joins don't crash
    df_finstore = df_finstore.withColumn("sourcesystemname", F.lit(""))
    print("⚠️ Finstore: neither 'bookedsystem' nor 'sourcesystemname' found — created empty column")

# --- ReconSubProduct classification ---
df_axis = df_axis.withColumn(
    "ReconSubProduct",
    F.when(F.col("SourceSystemName").isin(ETD_SYSTEMS), F.lit("ETD"))
     .when(F.col("SourceSystemName").isin(OTC_SYSTEMS), F.lit("OTC"))
     .otherwise(F.lit("OTC-Default"))
)

# --- DerivedSophisId: extract 3rd part of hyphen-separated SourceSystemTradeId ---
# Native Spark: split() + element_at() — no UDF needed
df_axis = df_axis.withColumn(
    "DerivedSophisId",
    F.when(
        F.col("SourceSystemName").isin(SOPHIS_SYSTEMS) &
        F.col("SourceSystemTradeId").isNotNull() &
        (F.size(F.split(F.col("SourceSystemTradeId"), "-")) >= 3),
        F.element_at(F.split(F.col("SourceSystemTradeId"), "-"), 3)
    ).otherwise(F.lit(""))
)

# --- DerivedDelta1Id: same extraction for DELTA1 systems ---
df_axis = df_axis.withColumn(
    "DerivedDelta1Id",
    F.when(
        F.col("SourceSystemName").isin(DELTA1_SYSTEMS) &
        F.col("SourceSystemTradeId").isNotNull() &
        (F.size(F.split(F.col("SourceSystemTradeId"), "-")) >= 3),
        F.element_at(F.split(F.col("SourceSystemTradeId"), "-"), 3)
    ).otherwise(F.lit(""))
)

# --- DerivedMasterbookId: from SDS mapping (broadcast join if available) ---
# SDS mapping schema: SourceSystemTradeId, MasterbookId, BookId
# When available, broadcast-join on SourceSystemTradeId to populate
# DerivedMasterbookId — this enables rules P5, P7, P13, P14.
if sds_available and df_sds_mapping is not None:
    df_sds_mapping = df_sds_mapping.select(
        F.col("SourceSystemTradeId").alias("_sds_trade_id"),
        F.col("MasterbookId").alias("_sds_masterbook_id"),
    ).dropDuplicates(["_sds_trade_id"])

    df_axis = df_axis.join(
        F.broadcast(df_sds_mapping),
        df_axis["SourceSystemTradeId"] == F.col("_sds_trade_id"),
        how="left",
    ).withColumn(
        "DerivedMasterbookId",
        F.coalesce(F.col("_sds_masterbook_id"), F.lit("")),
    ).drop("_sds_trade_id", "_sds_masterbook_id")

    print("DerivedMasterbookId: populated via SDS broadcast join "
          "(P5, P7, P13, P14 now active)")
else:
    df_axis = df_axis.withColumn("DerivedMasterbookId", F.lit(""))
    print("DerivedMasterbookId: SDS not available — set to '' "
          "(rules P5, P7, P13, P14 will be skipped)")

# --- Ensure amount columns are numeric ---
df_axis = df_axis.withColumn(AXIS_AMOUNT_COL, F.col(AXIS_AMOUNT_COL).cast("double"))
df_finstore = df_finstore.withColumn(FIN_AMOUNT_COL, F.col(FIN_AMOUNT_COL).cast("double"))

# --- Ensure string join columns are trimmed ---
string_cols_axis = [
    "SourceSystemTradeId", "BookId", "DerivedSophisId", "DerivedDelta1Id",
    "DerivedMasterbookId", "SourceSystemInstrumentId", "CounterpartyId"
]
for c in string_cols_axis:
    if c in df_axis.columns:
        df_axis = df_axis.withColumn(c, F.trim(F.col(c).cast("string")))

string_cols_fin = [
    "tradeid", "alternatetradeid1", "alternatetradeid2", "masterbookid",
    "tradingsystembook", "fissnumber", "instrumentid", "counterpartyid"
]
for c in string_cols_fin:
    if c in df_finstore.columns:
        df_finstore = df_finstore.withColumn(c, F.trim(F.col(c).cast("string")))

print("PRE-RECONCILIATION DERIVATIONS DEFINED (lazy — no Spark jobs triggered)")
print("Derivations will materialise when core tables are cached in the next section.")

---
## 🔹 Section 7: Core / Wide Split (Match Narrow, Enrich Wide)

**Critical performance optimisation:** only carry the ~15 columns needed for matching.
The remaining 100+ columns stay in `_wide` tables and are joined back at the end.

In [ ]:
# ============================================================
# CORE / WIDE SPLIT — minimise shuffle payload
# ============================================================
# ⚡ PERF: These .count() calls are INTENTIONAL — they materialise the cache.
#    This is the single point where all deferred counts are resolved.
#    Uses MEMORY_AND_DISK to spill gracefully if 4M+ rows don't fit in RAM.

# Axis core: only columns needed for matching
AXIS_CORE_COLS = [
    "axis_id", "SourceSystemName", "SourceSystemTradeId", "BookId",
    "CounterpartyId", AXIS_AMOUNT_COL, "SourceSystemInstrumentId",
    "DerivedSophisId", "DerivedDelta1Id", "DerivedMasterbookId",
    "ReconSubProduct",
]
# Filter to columns that actually exist
AXIS_CORE_COLS = [c for c in AXIS_CORE_COLS if c in df_axis.columns]

axis_core = df_axis.select(AXIS_CORE_COLS)
axis_wide = df_axis  # keep full DataFrame; we join by axis_id at the end

# Finstore core: only columns needed for matching
FIN_CORE_COLS = [
    "fin_id", "tradeid", "alternatetradeid1", "alternatetradeid2",
    "masterbookid", "tradingsystembook", "fissnumber", "instrumentid",
    "counterpartyid", FIN_AMOUNT_COL,
    "sourcesystemname",  # needed for P15 extra equi-join (v5.0)
    GREEDY_SAP_FILTER_COL,  # needed for greedy SAP account filter (v5.6)
]
FIN_CORE_COLS = [c for c in FIN_CORE_COLS if c in df_finstore.columns]

fin_core = df_finstore.select(FIN_CORE_COLS)
fin_wide = df_finstore  # keep full DataFrame; we join by fin_id at the end

# Cache core tables with MEMORY_AND_DISK — they are reused across all 15 rules + greedy
axis_core = axis_core.persist(StorageLevel.MEMORY_AND_DISK)
fin_core = fin_core.persist(StorageLevel.MEMORY_AND_DISK)

# Materialise cache + get counts (single Spark job per DF)
ORIGINAL_AXIS_COUNT = axis_core.count()
ORIGINAL_FINSTORE_COUNT = fin_core.count()

# Derive excluded count arithmetically (no extra Spark job)
if EXCLUDE_SOPHIS_DELTA1:
    # We need the full count — but we can get it from df_axis_full cheaply
    # since it's a simple CSV read that was already scanned above.
    # Use a single quick count on the excluded DF instead.
    ORIGINAL_AXIS_COUNT_FULL = ORIGINAL_AXIS_COUNT  # will be updated below
    excluded_count = 0  # placeholder
    # Compute full count once
    _full_axis_count = df_axis_full.count()
    ORIGINAL_AXIS_COUNT_FULL = _full_axis_count
    excluded_count = ORIGINAL_AXIS_COUNT_FULL - ORIGINAL_AXIS_COUNT
else:
    ORIGINAL_AXIS_COUNT_FULL = ORIGINAL_AXIS_COUNT
    excluded_count = 0

print(f"\n{'='*80}")
print("CORE TABLES CACHED — ALL DEFERRED COUNTS RESOLVED")
print(f"{'='*80}")
print(f"Axis (full):     {ORIGINAL_AXIS_COUNT_FULL:,} records")
print(f"Axis (in-scope): {ORIGINAL_AXIS_COUNT:,} records ({len(AXIS_CORE_COLS)} core cols)")
if EXCLUDE_SOPHIS_DELTA1:
    print(f"Axis (excluded): {excluded_count:,} records")
print(f"Finstore:        {ORIGINAL_FINSTORE_COUNT:,} records ({len(FIN_CORE_COLS)} core cols)")
print(f"\nAxis wide cols:    {len(df_axis.columns)}")
print(f"Finstore wide cols:{len(df_finstore.columns)}")

---
## ✅ LAYER 1: BRD DETERMINISTIC MATCHING (15 Rules)

### Architecture: Sequential Waterfall via `safe_rule_join()` (v5.0)

Rules execute **sequentially** in priority order (1 → 15).
After each rule, matched `axis_id`s are accumulated and anti-joined
from the next rule's input — exactly replicating the Pandas sequential
waterfall where matched trades are removed from the pool.

Each rule is wrapped by `safe_rule_join()` which provides:
- Input restriction (anti-join against already-matched axis_ids)
- System filtering, key cleaning, pre-join diagnostics
- Per-key Finstore cap, extra join constraints, top-K per axis
- Circuit breaker for explosion protection

Within a rule, all inner-join rows are preserved (1 axis → N finstores
for multi-leg instruments).  Pool removal uses distinct `axis_id` sets.

---
### 🔹 Section 8: safe_rule_join — Hardened Candidate Generation (v5.0)

Every BRD rule is wrapped by `safe_rule_join()` which provides:
1. **Input restriction**: anti-join against already-matched axis_ids (sequential waterfall)
2. **System filtering**: SOPHIS-only, ETD-only, or custom system lists
3. **Key cleaning**: null, empty, `nan`, `$` filtering
4. **Pre-join diagnostics + circuit breaker**: estimated rows → skip if > cap
5. **Per-key Finstore cap**: `MAX_FIN_PER_KEY` (bounds worst-case fan-out)
6. **Extra join constraints**: rule-specific additional equi-join predicates
7. **Top-K per axis**: deterministic ranking, configurable via `rule["top_k"]`
8. **Pair dedup**: no duplicate `(axis_id, fin_id)` edges

In [ ]:
# ============================================================
# CANDIDATE GENERATION — safe_rule_join wrapper (v5.0)
# ============================================================
# ARCHITECTURE:
#   safe_rule_join() wraps every BRD rule with:
#     1. INPUT RESTRICTION: anti-join against already-matched axis_ids
#        (sequential waterfall — each rule only sees unmatched Axis)
#     2. KEY CLEANING: null, empty, 'nan', '$' filtering
#     3. PRE-JOIN DIAGNOSTICS: cardinality, skew, estimated rows
#     4. CIRCUIT BREAKER: skip rule if est rows > MAX_CANDIDATES_PER_RULE
#     5. PER-KEY FINSTORE CAP: limit fan-out per join key value
#     6. EXTRA JOIN CONSTRAINTS: rule-specific additional equi-joins
#        (e.g., P15 adds SourceSystemName matching)
#     7. TOP-K PER AXIS: deterministic ranking to keep at most top_k
#        fin matches per axis within this rule
#     8. PAIR DEDUP: no duplicate (axis_id, fin_id) edges
#
# The candidate generation loop in Section 9 calls safe_rule_join()
# sequentially, accumulating matched axis_ids between rules.

# ── Guardrail thresholds (tunable) ──────────────────────────────────────
MAX_FIN_PER_KEY = 50            # Max Finstore rows kept per join-key value
MAX_CANDIDATES_PER_RULE = 10_000_000   # Circuit breaker: skip rule if est. rows > this
DEFAULT_TOP_K = 20              # Default: keep at most 20 fin matches per axis per rule

# Check once: does ANY Axis row have a non-empty DerivedMasterbookId?
HAS_DERIVED_MASTERBOOK = (
    axis_core
    .filter((F.col("DerivedMasterbookId").isNotNull()) & (F.col("DerivedMasterbookId") != ""))
    .limit(1)
    .count() > 0
)
print(f"DerivedMasterbookId populated: {HAS_DERIVED_MASTERBOOK}")

# Empty candidate schema (reused across skipped rules — created once)
_EMPTY_CANDIDATE_SCHEMA = T.StructType([
    T.StructField("axis_id", T.LongType()),
    T.StructField("fin_id", T.LongType()),
    T.StructField("priority", T.IntegerType()),
    T.StructField("category", T.StringType()),
    T.StructField("brd_priority", T.IntegerType()),
    T.StructField("description", T.StringType()),
    T.StructField("amount_diff", T.DoubleType()),
    T.StructField("amount_rel_diff", T.DoubleType()),
    T.StructField("key_strength", T.IntegerType()),
])

# Accumulator for per-rule diagnostics (populated during candidate generation)
_rule_diagnostics = []


def _filter_invalid_keys(df: DataFrame, keys: list) -> DataFrame:
    """Filter out rows with null, empty, 'nan', or '$'-containing key values."""
    for k in keys:
        df = df.filter(
            (F.col(k).isNotNull()) &
            (F.col(k) != "") &
            (F.col(k) != "nan") &
            (~F.col(k).contains("$"))
        )
    return df


def _compute_pre_join_diagnostics(
    a_sel: DataFrame,
    f_sel: DataFrame,
    join_cols: list,
    rule: dict,
) -> dict:
    """
    Compute pre-join risk diagnostics for a single rule.
    Returns a dict with cardinality, skew, and estimated join size.
    """
    priority = rule["priority"]
    desc = rule["description"]

    axis_freq = a_sel.groupBy(join_cols).agg(F.count("*").alias("axis_cnt"))
    fin_freq  = f_sel.groupBy(join_cols).agg(F.count("*").alias("fin_cnt"))

    key_cross = axis_freq.join(fin_freq, on=join_cols, how="inner")
    key_cross = key_cross.withColumn("cross_product", F.col("axis_cnt") * F.col("fin_cnt"))

    stats = key_cross.agg(
        F.sum("cross_product").alias("est_rows"),
        F.count("*").alias("overlap_keys"),
        F.max("axis_cnt").alias("max_axis_per_key"),
        F.max("fin_cnt").alias("max_fin_per_key"),
        F.max("cross_product").alias("max_cross_product"),
        F.sum("axis_cnt").alias("axis_rows_overlap"),
        F.sum("fin_cnt").alias("fin_rows_overlap"),
    ).first()

    est_rows = int(stats["est_rows"] or 0)
    diag = {
        "priority": priority,
        "description": desc,
        "category": rule["category"],
        "overlap_keys": int(stats["overlap_keys"] or 0),
        "est_rows": est_rows,
        "max_axis_per_key": int(stats["max_axis_per_key"] or 0),
        "max_fin_per_key": int(stats["max_fin_per_key"] or 0),
        "max_cross_product": int(stats["max_cross_product"] or 0),
        "axis_rows_overlap": int(stats["axis_rows_overlap"] or 0),
        "fin_rows_overlap": int(stats["fin_rows_overlap"] or 0),
    }

    top_keys = (
        key_cross
        .orderBy(F.col("cross_product").desc())
        .limit(20)
        .collect()
    )
    diag["top_keys"] = [
        {
            "key": str(row[join_cols[0]]) if len(join_cols) == 1
                   else "|".join(str(row[c]) for c in join_cols),
            "axis_cnt": row["axis_cnt"],
            "fin_cnt": row["fin_cnt"],
            "cross_product": row["cross_product"],
        }
        for row in top_keys
    ]

    return diag


def safe_rule_join(
    axis_df: DataFrame,
    fin_df: DataFrame,
    rule: dict,
    matched_axis_ids: DataFrame = None,
) -> DataFrame:
    """
    🛡️ safe_rule_join — hardened candidate generation for a single BRD rule.

    Wraps the join with:
      1. INPUT RESTRICTION: anti-join against matched_axis_ids (waterfall)
      2. SYSTEM FILTER: SOPHIS-only or ETD-only as configured
      3. KEY CLEANING: filter nulls, blanks, 'nan', '$'
      4. PRE-JOIN DIAGNOSTICS + CIRCUIT BREAKER
      5. PER-KEY FINSTORE CAP: MAX_FIN_PER_KEY
      6. EXTRA JOIN CONSTRAINTS: rule-specific additional equi-joins
      7. TOP-K PER AXIS: deterministic ranking, configurable via rule["top_k"]
      8. PAIR DEDUP: dropDuplicates(["axis_id", "fin_id"])

    Parameters
    ----------
    axis_df : DataFrame
        axis_core (full cached table).
    fin_df : DataFrame
        fin_core (full cached table).
    rule : dict
        Rule definition from MATCHING_RULES. Supports optional keys:
        - "system_filter": list of systems to restrict Axis to
        - "extra_axis_keys" / "extra_fin_keys": additional equi-join columns
        - "top_k": max fin matches per axis for this rule (default: DEFAULT_TOP_K)
    matched_axis_ids : DataFrame | None
        Distinct axis_ids already matched by higher-priority rules.
        If provided, these are anti-joined from axis_df before the rule runs.
    """
    priority = rule["priority"]
    axis_keys = rule["axis_keys"]
    fin_keys = rule["fin_keys"]
    top_k = rule.get("top_k", DEFAULT_TOP_K)

    # ── 1. INPUT RESTRICTION: remove already-matched Axis trades ─────────
    a = axis_df
    if matched_axis_ids is not None:
        a = a.join(matched_axis_ids, on="axis_id", how="left_anti")

    # ── 2. SYSTEM FILTER ─────────────────────────────────────────────────
    if rule["filter_sophis_only"]:
        a = a.filter(F.col("SourceSystemName").isin(SOPHIS_SYSTEMS))
    elif rule.get("system_filter"):
        a = a.filter(F.col("SourceSystemName").isin(rule["system_filter"]))

    # ── Skip if DerivedMasterbookId required but not populated ───────────
    if rule["requires_derived_masterbook"] and not HAS_DERIVED_MASTERBOOK:
        _rule_diagnostics.append({
            "priority": priority, "description": rule["description"],
            "category": rule["category"], "status": "SKIPPED (no DerivedMasterbookId)",
            "est_rows": 0, "overlap_keys": 0, "actual_rows": 0,
            "max_axis_per_key": 0, "max_fin_per_key": 0, "max_cross_product": 0,
            "axis_rows_overlap": 0, "fin_rows_overlap": 0, "top_keys": [],
            "axis_input_count": 0,
        })
        return spark.createDataFrame([], _EMPTY_CANDIDATE_SCHEMA)

    # ── 3. Project to narrow schema ──────────────────────────────────────
    # Primary join keys
    a_cols = list(dict.fromkeys(["axis_id", AXIS_AMOUNT_COL, "SourceSystemName"] + axis_keys))
    a_sel = a.select([F.col(c) for c in a_cols if c in a.columns])

    f_cols = list(dict.fromkeys(["fin_id", FIN_AMOUNT_COL] + fin_keys))
    f_sel = fin_df.select([F.col(c) for c in f_cols if c in fin_df.columns])

    # Extra join keys (optional — for P15-style tightening)
    extra_axis_keys = rule.get("extra_axis_keys", [])
    extra_fin_keys = rule.get("extra_fin_keys", [])
    for ek in extra_axis_keys:
        if ek not in [c for c in a_sel.columns] and ek in a.columns:
            a_sel = a_sel.withColumn(ek, a[ek])
    for ek in extra_fin_keys:
        if ek not in [c for c in f_sel.columns] and ek in fin_df.columns:
            f_sel = f_sel.withColumn(ek, fin_df[ek])

    # ── KEY CLEANING ─────────────────────────────────────────────────────
    a_sel = _filter_invalid_keys(a_sel, axis_keys)
    f_sel = _filter_invalid_keys(f_sel, fin_keys)

    # ── Rename primary keys to common join names ─────────────────────────
    for i, ak in enumerate(axis_keys):
        a_sel = a_sel.withColumnRenamed(ak, f"_jk{i}")
    for i, fk in enumerate(fin_keys):
        f_sel = f_sel.withColumnRenamed(fk, f"_jk{i}")

    join_cols = [f"_jk{i}" for i in range(len(axis_keys))]

    # ── 4. PRE-JOIN DIAGNOSTICS + CIRCUIT BREAKER ────────────────────────
    diag = _compute_pre_join_diagnostics(a_sel, f_sel, join_cols, rule)
    est_rows = diag["est_rows"]

    # Track how many axis rows enter this rule (for reporting)
    axis_input_count = a_sel.select("axis_id").distinct().count()
    diag["axis_input_count"] = axis_input_count

    if est_rows > MAX_CANDIDATES_PER_RULE:
        diag["status"] = f"⚠️ CAPPED (est {est_rows:,} > limit {MAX_CANDIDATES_PER_RULE:,})"
        print(f"  🛡️ Rule P{priority}: est_rows={est_rows:,} > cap={MAX_CANDIDATES_PER_RULE:,} "
              f"→ applying per-key Finstore cap ({MAX_FIN_PER_KEY}) + top_k={top_k}")
    else:
        diag["status"] = f"OK (est {est_rows:,})"

    # ── 5. PER-KEY FINSTORE CAP ──────────────────────────────────────────
    w_fin_per_key = Window.partitionBy(join_cols).orderBy(F.col(FIN_AMOUNT_COL).asc_nulls_last())
    f_sel = (
        f_sel
        .withColumn("_fin_rn", F.row_number().over(w_fin_per_key))
        .filter(F.col("_fin_rn") <= MAX_FIN_PER_KEY)
        .drop("_fin_rn")
    )

    # ── JOIN (primary keys) ──────────────────────────────────────────────
    joined = a_sel.join(f_sel, on=join_cols, how="inner")

    # ── 6. EXTRA JOIN CONSTRAINTS (post-join filter) ─────────────────────
    # These act as STRICT additional equi-join predicates applied after the
    # primary key join. They narrow M:M without requiring a compound key.
    # v5.1 fix: use case-insensitive UPPER() equality with NO null bypass.
    # Previously, null/empty on either side would pass through, defeating
    # the purpose of the constraint (e.g., P15 SourceSystemName tightening).
    for eax, efk in zip(extra_axis_keys, extra_fin_keys):
        ax_col = eax  # already in a_sel (added above)
        fk_col = efk  # already in f_sel
        if ax_col in joined.columns and fk_col in joined.columns:
            joined = joined.filter(
                F.upper(F.col(ax_col)) == F.upper(F.col(fk_col))
            )

    # ── 8. PAIR DEDUP ────────────────────────────────────────────────────
    joined = joined.dropDuplicates(["axis_id", "fin_id"])

    # ── Compute amount_diff ──────────────────────────────────────────────
    joined = joined.withColumn(
        "amount_diff",
        F.abs(F.col(AXIS_AMOUNT_COL) - F.col(FIN_AMOUNT_COL))
    )
    joined = joined.withColumn(
        "amount_rel_diff",
        F.col("amount_diff") / F.greatest(F.abs(F.col(AXIS_AMOUNT_COL)), F.lit(1e-9))
    )
    key_strength = len(axis_keys) + len(extra_axis_keys)
    joined = joined.withColumn("key_strength", F.lit(key_strength).cast("int"))

    # ── 7. TOP-K PER AXIS: deterministic ranking ─────────────────────────
    # Ranking: smallest amount_diff → highest key_strength → smallest fin_id
    w_topk = Window.partitionBy("axis_id").orderBy(
        F.col("amount_diff").asc_nulls_last(),
        F.col("key_strength").desc(),
        F.col("fin_id").asc(),
    )
    joined = (
        joined
        .withColumn("_topk_rn", F.row_number().over(w_topk))
        .filter(F.col("_topk_rn") <= top_k)
        .drop("_topk_rn")
    )

    # ── Record actual output rows in diagnostics ─────────────────────────
    actual_rows = joined.count()
    diag["actual_rows"] = actual_rows
    _rule_diagnostics.append(diag)

    print(f"    → P{priority}: axis_in={axis_input_count:,}, "
          f"est={est_rows:,}, actual={actual_rows:,} (top_k={top_k})")

    # ── Output standardised candidate schema ─────────────────────────────
    return joined.select(
        F.col("axis_id"),
        F.col("fin_id"),
        F.lit(priority).cast("int").alias("priority"),
        F.lit(rule["category"]).alias("category"),
        F.lit(rule["brd_priority"]).cast("int").alias("brd_priority"),
        F.lit(rule["description"]).alias("description"),
        F.col("amount_diff"),
        F.col("amount_rel_diff"),
        F.col("key_strength"),
    )


print("safe_rule_join() defined (v5.0 — sequential waterfall with anti-explosion guardrails).")
print(f"  MAX_FIN_PER_KEY = {MAX_FIN_PER_KEY}")
print(f"  MAX_CANDIDATES_PER_RULE = {MAX_CANDIDATES_PER_RULE:,}")
print(f"  DEFAULT_TOP_K = {DEFAULT_TOP_K}")

---
### 🔹 Section 9: Sequential Waterfall — Generate All BRD Candidates (v5.0)

Rules execute in priority order. After each rule, matched axis_ids accumulate
so the next rule only processes truly unmatched trades.

In [ ]:
# ============================================================
# GENERATE BRD CANDIDATES — SEQUENTIAL WATERFALL (v5.0)
# ============================================================
# Unlike v4.x which ran all 15 rules against the FULL pool in parallel
# and then resolved best-rule-per-axis, v5.0 runs rules SEQUENTIALLY
# in priority order.  After each rule, newly matched axis_ids are
# accumulated and anti-joined from the next rule's input.
#
# This replicates the Pandas sequential waterfall EXACTLY:
#   - Each rule only sees unmatched Axis trades
#   - Within a rule, all inner-join rows preserved (1:N multi-leg)
#   - matched axis_ids accumulate across rules
#
# Benefits:
#   - P15 now only sees Axis trades NOT matched by P1-P14
#   - Eliminates cross-rule explosion (P15 used to process 3.29M Axis)
#   - Each rule's diagnostic shows actual input size after pool removal

print(f"\n{'='*80}")
print("LAYER 1: GENERATING BRD CANDIDATES (Sequential Waterfall v5.0)")
print(f"{'='*80}")

candidate_dfs = []
matched_axis_ids = None  # Accumulator: distinct axis_ids matched so far

for rule in sorted(MATCHING_RULES, key=lambda r: r["priority"]):
    print(f"\n  Rule P{rule['priority']:2d} ({rule['category']:6s} #{rule['brd_priority']}): "
          f"{rule['description']}")

    cand = safe_rule_join(axis_core, fin_core, rule, matched_axis_ids)
    candidate_dfs.append(cand)

    # Accumulate matched axis_ids for waterfall pool removal
    new_axis_ids = cand.select("axis_id").distinct()
    if matched_axis_ids is None:
        matched_axis_ids = new_axis_ids
    else:
        matched_axis_ids = matched_axis_ids.union(new_axis_ids).distinct()

    # Checkpoint every 5 rules to break lineage chains and prevent OOM
    if rule["priority"] % 5 == 0:
        matched_axis_ids = matched_axis_ids.persist(StorageLevel.MEMORY_AND_DISK)

# Union all candidate edges
candidates_layer1 = reduce(DataFrame.unionByName, candidate_dfs)

# Persist — materialisation happens when resolution reads it
candidates_layer1 = candidates_layer1.persist(StorageLevel.MEMORY_AND_DISK)

# Report total accumulated matches
total_matched_axis_waterfall = matched_axis_ids.count() if matched_axis_ids is not None else 0
print(f"\n{'─'*60}")
print(f"Sequential waterfall complete.")
print(f"  Total distinct Axis matched across all rules: {total_matched_axis_waterfall:,}")
print(f"  Candidates unioned and persisted.")

---
### 🔍 Section 9b: Pre-Join Risk Diagnostics — Sequential Waterfall (v5.0)

For **every** BRD rule, `safe_rule_join()` computed:
- **Axis input count** (after sequential pool removal — shows how much the waterfall reduced input)
- Overlapping key count, estimated join rows, **actual output rows** (after top_k + guardrails)
- Maximum per-key multiplicity on each side (skew indicators)
- Top-20 explosion-driver keys
- Pool removal effectiveness chart

In [ ]:
# ============================================================
# SECTION 9b: PRE-JOIN RISK DIAGNOSTICS — ALL 15 BRD RULES (v5.0)
# ============================================================
# _rule_diagnostics was populated during safe_rule_join().
# Now includes: axis_input_count (after pool removal), actual_rows (post top-k).

import datetime as _dt

print(f"\n{'='*80}")
print("🔍 PRE-JOIN RISK DIAGNOSTICS — ALL 15 BRD RULES (Sequential Waterfall v5.0)")
print(f"   Timestamp: {_dt.datetime.now().isoformat()}")
print(f"   Guardrails: MAX_FIN_PER_KEY={MAX_FIN_PER_KEY}, "
      f"MAX_CANDIDATES_PER_RULE={MAX_CANDIDATES_PER_RULE:,}, "
      f"DEFAULT_TOP_K={DEFAULT_TOP_K}")
print(f"{'='*80}")

# ── Summary table ───────────────────────────────────────────────────────
print(f"\n{'─'*140}")
print(f"{'P':>3s} {'Cat':>6s} {'Description':<55s} {'AxisIn':>8s} "
      f"{'OvlpK':>8s} {'EstRows':>12s} {'ActRows':>10s} "
      f"{'MxAxK':>7s} {'MxFnK':>7s} {'MxX':>10s} {'Status'}")
print(f"{'─'*140}")

rules_at_risk = []
for d in sorted(_rule_diagnostics, key=lambda x: x["priority"]):
    p = d["priority"]
    est = d["est_rows"]
    actual = d.get("actual_rows", 0)
    axis_in = d.get("axis_input_count", 0)
    flag = "🔴" if est > MAX_CANDIDATES_PER_RULE else ("🟡" if est > 1_000_000 else "🟢")

    # Compression ratio: how much the guardrails reduced output
    compression = f"({actual/est*100:.0f}%)" if est > 0 else ""

    print(f"{p:>3d} {d['category']:>6s} {d['description'][:55]:<55s} "
          f"{axis_in:>8,} {d['overlap_keys']:>8,} {est:>12,} "
          f"{actual:>10,} {d['max_axis_per_key']:>7,} "
          f"{d['max_fin_per_key']:>7,} {d['max_cross_product']:>10,} "
          f"{flag} {d.get('status','')} {compression}")
    if est > 1_000_000:
        rules_at_risk.append(d)

print(f"{'─'*140}")
print(f"Legend: 🟢 < 1M est | 🟡 1M-{MAX_CANDIDATES_PER_RULE/1e6:.0f}M | "
      f"🔴 > {MAX_CANDIDATES_PER_RULE/1e6:.0f}M")
print(f"  AxisIn = Axis rows entering rule (after sequential pool removal)")
print(f"  ActRows = actual output rows (after top_k + guardrails)")

# ── Pool removal effectiveness ──────────────────────────────────────────
print(f"\n{'='*80}")
print("📉 SEQUENTIAL WATERFALL POOL REMOVAL EFFECTIVENESS")
print(f"{'='*80}")
for d in sorted(_rule_diagnostics, key=lambda x: x["priority"]):
    p = d["priority"]
    axis_in = d.get("axis_input_count", 0)
    reduction = ORIGINAL_AXIS_COUNT - axis_in if axis_in > 0 else 0
    pct = reduction / ORIGINAL_AXIS_COUNT * 100 if ORIGINAL_AXIS_COUNT > 0 else 0
    bar = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))
    print(f"  P{p:>2d}: {axis_in:>10,} axis in ({pct:>5.1f}% removed) {bar}")

# ── Detailed breakdown for at-risk rules ────────────────────────────────
if rules_at_risk:
    print(f"\n{'='*80}")
    print(f"⚠️  DETAILED BREAKDOWN: {len(rules_at_risk)} RULE(S) WITH EST > 1M ROWS")
    print(f"{'='*80}")

    for d in rules_at_risk:
        p = d["priority"]
        actual = d.get("actual_rows", 0)
        print(f"\n── Rule P{p}: {d['description']} ──")
        print(f"   Category:             {d['category']}")
        print(f"   Axis input (post-WF): {d.get('axis_input_count', 0):,}")
        print(f"   Overlapping keys:     {d['overlap_keys']:,}")
        print(f"   Axis rows in overlap: {d['axis_rows_overlap']:,}")
        print(f"   Fin rows in overlap:  {d['fin_rows_overlap']:,}")
        print(f"   ESTIMATED join rows:  {d['est_rows']:,}")
        print(f"   ACTUAL output rows:   {actual:,}")
        if d['est_rows'] > 0:
            print(f"   Compression ratio:    {actual/d['est_rows']*100:.1f}%")
        print(f"   Status:               {d.get('status','')}")

        if d.get("top_keys"):
            print(f"\n   Top-20 explosion-driver keys:")
            for i, tk in enumerate(d["top_keys"], 1):
                print(f"     {i:>2d}. {tk['key']:<40s} "
                      f"axis={tk['axis_cnt']:>8,}  fin={tk['fin_cnt']:>8,}  "
                      f"cross={tk['cross_product']:>12,}")
else:
    print("\n✅ No rules estimated > 1M rows. Sequential waterfall + guardrails working well.")

# ── P15-specific diagnosis ──────────────────────────────────────────────
p15_diag = next((d for d in _rule_diagnostics if d["priority"] == 15), None)
if p15_diag:
    print(f"\n{'='*80}")
    print("📋 P15 DIAGNOSIS (previously 101M+ rows)")
    print(f"{'='*80}")
    print(f"  Axis input (after P1-P14 removed): {p15_diag.get('axis_input_count', 0):>12,}")
    print(f"  Estimated rows (post pool removal): {p15_diag['est_rows']:>12,}")
    print(f"  Actual output rows (post top_k):    {p15_diag.get('actual_rows', 0):>12,}")
    print(f"  Status: {p15_diag.get('status', '')}")
    print()
    print("  v5.0 MITIGATIONS APPLIED:")
    print("  ✅ 1. Sequential waterfall: only unmatched Axis enters P15")
    print("  ✅ 2. ETD system filter: only ETD systems participate")
    print("  ✅ 3. Extra equi-join: SourceSystemName must match")
    print("  ✅ 4. top_k=5: max 5 fin matches per axis")
    print("  ✅ 5. Per-key Finstore cap: MAX_FIN_PER_KEY={MAX_FIN_PER_KEY}")
    print(f"{'='*80}")

---
### 🔹 Section 10: BRD Match Resolution — Sequential Waterfall (v5.0)

**v5.0 change:** Rules now execute **sequentially** via `safe_rule_join()`.
Each rule only sees Axis trades NOT matched by higher-priority rules
(anti-join pool removal). This eliminates cross-rule explosion and
exactly replicates the Pandas sequential waterfall.

Within the winning rule, all inner-join rows are preserved (1 axis → N finstores).

The "best-rule-per-axis" filter below is a **safety net** — with correct
sequential execution it should not remove any rows.

In [ ]:
# ============================================================
# BRD MATCH RESOLUTION — Waterfall Parity (v5.0)
# ============================================================
# With sequential execution (v5.0), each axis_id participates in
# exactly ONE rule (the first one that matches it). The best-rule
# filter below is a SAFETY NET — it shouldn't change any rows,
# but protects against edge cases.
#
# Within that winning rule, all inner-join rows are preserved
# (1 axis → N finstores for multi-leg instruments).

# ── resolve_matches: DEPRECATED (v5.4) ──────────────────────────────────
# This function is no longer called.  Greedy strategies use iterative
# shrinking-pool loops (v5.3+).  BRD uses a window-based best-priority
# filter directly.  Kept for reference only — will be removed in v6.
def resolve_matches(candidates: DataFrame, max_iter: int = None) -> DataFrame:
    """
    Enforce EXCLUSIVE 1:1 matching between axis_id and fin_id.

    Pandas greedy uses a running set (matched_finstore_indices_greedy) to
    prevent the same Finstore record from being matched by multiple Axis
    trades.  The v5.0 PySpark version only picked 1 best fin per axis but
    did NOT prevent the same fin from serving multiple axes → inflated
    match rate.

    v5.1 fix: two-pass "stable-marriage" iteration:
      Pass A – best fin per axis (window on axis_id, row_number)
      Pass B – best axis per fin (window on fin_id, row_number)
    Repeat until convergence or max_iter.
    """
    if max_iter is None:
        max_iter = GREEDY_MAX_ITER

    has_key_strength = "key_strength" in candidates.columns
    ordering = [F.col("priority").asc()]
    if has_key_strength:
        ordering.append(F.col("key_strength").desc())
    ordering.append(F.col("amount_diff").asc_nulls_last())

    current = candidates
    for i in range(max_iter):
        prev_count = current.count()

        # Pass A: best fin per axis
        w_axis = Window.partitionBy("axis_id").orderBy(*ordering, F.col("fin_id").asc())
        current = (
            current
            .withColumn("_rn_axis", F.row_number().over(w_axis))
            .filter(F.col("_rn_axis") == 1)
            .drop("_rn_axis")
        )

        # Pass B: best axis per fin (prevents fin reuse)
        w_fin = Window.partitionBy("fin_id").orderBy(*ordering, F.col("axis_id").asc())
        current = (
            current
            .withColumn("_rn_fin", F.row_number().over(w_fin))
            .filter(F.col("_rn_fin") == 1)
            .drop("_rn_fin")
        )

        new_count = current.count()
        if new_count == prev_count:
            print(f"    resolve_matches converged in {i+1} iteration(s): {new_count:,} exclusive pairs")
            break
    else:
        print(f"    resolve_matches: reached max_iter={max_iter}, {new_count:,} pairs (may not be fully exclusive)")

    return current


# ── BRD resolution: safety-net best-rule per axis ────────────────────────
print("BRD resolution: waterfall parity (sequential v5.0 + safety-net best-rule filter)...")

# Safety net: keep best (lowest) priority per axis_id
w_best_priority = Window.partitionBy("axis_id")
brd_matches = (
    candidates_layer1
    .withColumn("_best_priority", F.min("priority").over(w_best_priority))
    .filter(F.col("priority") == F.col("_best_priority"))
    .drop("_best_priority")
)

# Check if the safety net actually removed any rows
pre_safety_count = candidates_layer1.count()

# Add match layer label + auditability columns
brd_matches = (
    brd_matches
    .withColumn("MatchLayer", F.lit("BRD"))
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
    .withColumn("rule_version", F.lit(RULE_VERSION))
    .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
)

brd_matches = brd_matches.persist(StorageLevel.MEMORY_AND_DISK)
brd_match_rows = brd_matches.count()

# Check safety net effect
safety_net_removed = pre_safety_count - brd_match_rows
if safety_net_removed > 0:
    print(f"  ⚠️ Safety net removed {safety_net_removed:,} cross-rule duplicate rows")
else:
    print(f"  ✅ Safety net: no cross-rule duplicates (sequential waterfall working correctly)")

brd_unique_axis = brd_matches.select("axis_id").distinct().count()
brd_unique_fin  = brd_matches.select("fin_id").distinct().count()
brd_match_rate = brd_unique_axis / ORIGINAL_AXIS_COUNT * 100

print(f"\n{'='*80}")
print("LAYER 1 (BRD) RESULTS  —  Sequential Waterfall v5.0")
print(f"{'='*80}")
print(f"Total match rows:    {brd_match_rows:,}  (within-rule many-to-many)")
print(f"Unique Axis matched: {brd_unique_axis:,}  (each axis → exactly 1 rule)")
print(f"Unique Finstore used:{brd_unique_fin:,}")
print(f"BRD Match Rate (unique axis): {brd_match_rate:.2f}%")
print(f"Remaining unmatched: {ORIGINAL_AXIS_COUNT - brd_unique_axis:,}")

# Breakdown by rule
print("\nMatches by rule:")
brd_matches.groupBy("priority", "category", "description").count() \
    .orderBy("priority").show(20, truncate=False)

---
### 🔹 Section 10c: BRD Checkpoint Save (v5.4)

Save BRD deterministic results **before** greedy starts so they survive
greedy failures (OOM, timeout, cluster preemption).

Saved tables:
- `_brd_checkpoint/brd_matches` — all BRD match rows
- `_brd_checkpoint/brd_unmatched_axis` — axis trades not matched by BRD
- `_brd_checkpoint/brd_unmatched_fin` — finstore trades not matched by BRD

Controlled by `SAVE_BRD_CHECKPOINT` in the config cell.

In [ ]:
# ============================================================
# BRD CHECKPOINT SAVE  (v5.4)
# ============================================================
# Persists BRD results to Delta BEFORE greedy starts.
# If greedy crashes, you still have full deterministic results.

if SAVE_BRD_CHECKPOINT:
    print(f"Saving BRD checkpoint to: {BRD_CHECKPOINT_DIR}")
    print(f"{'-'*60}")

    # BRD matches
    brd_matches.write.format("delta").mode("overwrite").save(f"{BRD_CHECKPOINT_DIR}/brd_matches")
    print(f"✅ Saved: brd_matches ({brd_match_rows:,} rows)")

    # Unmatched pools (before greedy filtering)
    _chk_matched_axis = brd_matches.select("axis_id").distinct()
    _chk_matched_fin  = brd_matches.select("fin_id").distinct()
    _chk_unmatched_axis = axis_core.join(_chk_matched_axis, on="axis_id", how="left_anti")
    _chk_unmatched_fin  = fin_core.join(_chk_matched_fin, on="fin_id", how="left_anti")

    _chk_unmatched_axis.write.format("delta").mode("overwrite").save(f"{BRD_CHECKPOINT_DIR}/brd_unmatched_axis")
    _chk_unmatched_fin.write.format("delta").mode("overwrite").save(f"{BRD_CHECKPOINT_DIR}/brd_unmatched_fin")
    print(f"✅ Saved: brd_unmatched_axis, brd_unmatched_fin")

    # Save metadata
    import json
    _chk_meta = {
        "run_id": RUN_ID, "batch_id": BATCH_ID, "rule_version": RULE_VERSION,
        "brd_match_rows": brd_match_rows, "brd_unique_axis": brd_unique_axis,
        "brd_unique_fin": brd_unique_fin, "brd_match_rate": brd_match_rate,
        "original_axis_count": ORIGINAL_AXIS_COUNT,
        "original_finstore_count": ORIGINAL_FINSTORE_COUNT,
    }
    try:
        dbutils.fs.put(f"{BRD_CHECKPOINT_DIR}/_metadata.json", json.dumps(_chk_meta, indent=2), overwrite=True)
    except NameError:
        import os
        _chk_path = BRD_CHECKPOINT_DIR.replace("dbfs:", "/dbfs") if BRD_CHECKPOINT_DIR.startswith("dbfs:") else BRD_CHECKPOINT_DIR
        os.makedirs(_chk_path, exist_ok=True)
        with open(os.path.join(_chk_path, "_metadata.json"), "w") as f:
            json.dump(_chk_meta, f, indent=2)
    print(f"✅ Saved: _metadata.json")

    print(f"\n{'='*60}")
    print("BRD CHECKPOINT COMPLETE — greedy can fail safely.")
    print(f"{'='*60}")
else:
    print("BRD checkpoint SKIPPED (SAVE_BRD_CHECKPOINT=False)")

---
### 🔍 Section 10b: POC-Style Match Validation & Multi-Leg Reporting

Mirrors the Pandas POC validation to prove correctness at scale:
1. **Pair uniqueness**: no `(axis_id, fin_id)` pair repeats
2. **1-to-N distribution**: histogram of Finstore legs per Axis (multi-leg swaps are expected)
3. **Top Axis** with most Finstore legs
4. **Per-rule explosion check**: match rows vs unique Axis per rule (detects M:M within a rule)
5. **Reconciliation identity**: `unique_matched_axis + unmatched_axis == ORIGINAL_AXIS_COUNT`

In [ ]:
# ============================================================
# SECTION 10b: POC-STYLE MATCH VALIDATION & MULTI-LEG REPORTING (v5.0)
# ============================================================

print(f"\n{'='*80}")
print("🔍 POC-STYLE MATCH VALIDATION — BRD Layer (v5.0)")
print(f"{'='*80}")

# ── 1. Pair uniqueness: no (axis_id, fin_id) pair repeats ──────────────
pair_total = brd_match_rows  # already computed
pair_distinct = brd_matches.select("axis_id", "fin_id").distinct().count()
pair_dupes = pair_total - pair_distinct

print(f"\n── 1. Pair Uniqueness ──")
print(f"  Total BRD match rows:        {pair_total:>12,}")
print(f"  Distinct (axis_id, fin_id):  {pair_distinct:>12,}")
print(f"  Duplicate pairs:             {pair_dupes:>12,}")
if pair_dupes == 0:
    print("  ✅ PASSED — no (axis_id, fin_id) pair repeats")
else:
    print(f"  ⚠️  WARNING — {pair_dupes:,} duplicate pairs detected!")

# ── 2. Cross-rule uniqueness (sequential waterfall guarantee) ───────────
print(f"\n── 2. Sequential Waterfall Guarantee ──")
axis_rule_counts = (
    brd_matches
    .select("axis_id", "priority")
    .distinct()
    .groupBy("axis_id")
    .agg(F.countDistinct("priority").alias("n_rules"))
    .filter(F.col("n_rules") > 1)
)
multi_rule_axis = axis_rule_counts.count()
if multi_rule_axis == 0:
    print("  ✅ PASSED — every axis_id matched by exactly ONE rule")
    print("     (sequential waterfall with anti-join pool removal working correctly)")
else:
    print(f"  ⚠️  WARNING — {multi_rule_axis:,} axis_id(s) matched by MULTIPLE rules!")
    axis_rule_counts.orderBy(F.col("n_rules").desc()).show(10, truncate=False)

# ── 3. 1-to-N distribution: Finstore legs per Axis ─────────────────────
legs_per_axis = (
    brd_matches
    .groupBy("axis_id")
    .agg(F.count("fin_id").alias("n_legs"))
)

legs_distribution = (
    legs_per_axis
    .groupBy("n_legs")
    .agg(F.count("*").alias("axis_count"))
    .orderBy("n_legs")
)

print(f"\n── 3. 1-to-N Distribution (Finstore legs per Axis) ──")
legs_distribution.show(30, truncate=False)

legs_stats = legs_per_axis.agg(
    F.count("*").alias("total_axis"),
    F.sum(F.when(F.col("n_legs") == 1, 1).otherwise(0)).alias("axis_1to1"),
    F.sum(F.when(F.col("n_legs") > 1, 1).otherwise(0)).alias("axis_multi_leg"),
    F.max("n_legs").alias("max_legs"),
    F.avg("n_legs").alias("avg_legs"),
    F.expr("percentile_approx(n_legs, 0.5)").alias("median_legs"),
    F.expr("percentile_approx(n_legs, 0.95)").alias("p95_legs"),
    F.expr("percentile_approx(n_legs, 0.99)").alias("p99_legs"),
).first()

print(f"  Summary:")
print(f"    Total unique Axis matched:  {legs_stats['total_axis']:>10,}")
print(f"    1:1 matches:                {legs_stats['axis_1to1']:>10,}  "
      f"({legs_stats['axis_1to1']/max(legs_stats['total_axis'],1)*100:.1f}%)")
print(f"    Multi-leg (2+):             {legs_stats['axis_multi_leg']:>10,}  "
      f"({legs_stats['axis_multi_leg']/max(legs_stats['total_axis'],1)*100:.1f}%)")
print(f"    Max legs per Axis:          {legs_stats['max_legs']:>10,}")
print(f"    Avg legs per Axis:          {legs_stats['avg_legs']:>10.2f}")
print(f"    Median legs:                {legs_stats['median_legs']:>10,}")
print(f"    P95 legs:                   {legs_stats['p95_legs']:>10,}")
print(f"    P99 legs:                   {legs_stats['p99_legs']:>10,}")

# ── 4. Top-20 Axis trades with the most Finstore legs ──────────────────
print(f"\n── 4. Top-20 Axis with Most Finstore Legs ──")
top_multi_leg = (
    legs_per_axis
    .orderBy(F.col("n_legs").desc())
    .limit(20)
    .join(
        brd_matches.select("axis_id", "priority", "category", "description").distinct(),
        on="axis_id",
        how="left"
    )
)
top_multi_leg.show(20, truncate=False)

# ── 5. Per-rule metrics ────────────────────────────────────────────────
print(f"\n── 5. Per-Rule Match Quality Metrics ──")
per_rule_stats = (
    brd_matches
    .groupBy("priority", "category", "description")
    .agg(
        F.count("*").alias("match_rows"),
        F.countDistinct("axis_id").alias("unique_axis"),
        F.countDistinct("fin_id").alias("unique_fin"),
        F.avg("amount_diff").alias("avg_amount_diff"),
        F.avg("amount_rel_diff").alias("avg_rel_diff"),
        F.expr("percentile_approx(amount_diff, 0.5)").alias("median_amount_diff"),
        F.max("amount_diff").alias("max_amount_diff"),
    )
    .withColumn("rows_per_axis", F.round(F.col("match_rows") / F.col("unique_axis"), 2))
    .orderBy("priority")
)
per_rule_stats.show(20, truncate=False)

high_ratio_rules = per_rule_stats.filter(F.col("rows_per_axis") > 5.0).collect()
if high_ratio_rules:
    print("  ⚠️  Rules with rows_per_axis > 5.0:")
    for r in high_ratio_rules:
        print(f"     P{r['priority']} ({r['category']}): {r['match_rows']:,} rows / "
              f"{r['unique_axis']:,} axis = {r['rows_per_axis']:.1f}x  — {r['description']}")
else:
    print("  ✅ All rules have rows_per_axis ≤ 5.0")

# ── 6. Duplicate legitimacy check ──────────────────────────────────────
# Legitimate duplicates: same axis_id matched to multiple fin_ids
# (multi-leg swaps/structures). Illegitimate: same (axis_id, fin_id) pair.
print(f"\n── 6. Duplicate Legitimacy Check ──")
total_multi_leg_rows = brd_match_rows - brd_unique_axis
print(f"  Total 'extra' rows (multi-leg):    {total_multi_leg_rows:>10,}")
print(f"  These are Axis trades matched to 2+ Finstore records via the SAME rule.")
if pair_dupes == 0:
    print(f"  ✅ ALL multi-leg rows are LEGITIMATE (distinct axis_id × fin_id pairs)")
else:
    print(f"  ⚠️  {pair_dupes:,} are TRUE DUPLICATES (same axis_id × fin_id pair)")

# ── 7. Reconciliation identity ─────────────────────────────────────────
print(f"\n── 7. BRD Reconciliation Identity ──")
print(f"  ORIGINAL_AXIS_COUNT:     {ORIGINAL_AXIS_COUNT:>12,}")
print(f"  BRD unique axis matched: {brd_unique_axis:>12,}")
brd_unmatched_count = ORIGINAL_AXIS_COUNT - brd_unique_axis
print(f"  BRD unmatched axis:      {brd_unmatched_count:>12,}")
print(f"  Sum (matched+unmatched): {brd_unique_axis + brd_unmatched_count:>12,}")
if brd_unique_axis + brd_unmatched_count == ORIGINAL_AXIS_COUNT:
    print("  ✅ Reconciliation PASSED")
else:
    print("  ❌ Reconciliation FAILED — numbers don't add up!")

print(f"\n{'='*80}")
print("POC VALIDATION COMPLETE (v5.0)")
print(f"{'='*80}")

---
### 🔹 Section 11b: Layer 2 — Greedy / Probabilistic Matching

**Key Spark optimisations vs. Pandas (v5.3 — iterative shrinking-pool):**
- No Python `iterrows()` loops — everything is join + window-based
- **Iterative pattern** mirrors Pandas' running exclusion sets:
  join → best-1 per axis → best-1 per fin → commit batch → shrink pools → repeat
- Strategy 1: blocked on `counterpartyid` + 1% tolerance (matches Pandas groupby logic)
- Strategy 2: blocked on amount buckets (±1 bucket) + 0.1% strict tolerance
- Avoids O(A×F) candidate explosion from single-join approach

---
### 🔹 Section 11: Compute Unmatched Pools for Greedy

In [ ]:
# ============================================================
# COMPUTE UNMATCHED POOLS  (v5.6)
# ============================================================
# ⚡ PERF: No .count() — pools are lazy. They materialise when
#    greedy strategies read them (via their own cache).
#
# brd_matches may have within-rule many-to-many, so we use DISTINCT
# axis_id / fin_id for pool removal — identical to Pandas
# set(matched[idx].unique()).
#
# v5.6: SAP Account filter on Finstore pool to reduce greedy search space.
# v5.1: Added MIN_GREEDY_AMOUNT filter to prevent near-zero false positives.

# Distinct Axis / Fin IDs matched in Layer 1
matched_axis_ids = brd_matches.select("axis_id").distinct()
matched_fin_ids  = brd_matches.select("fin_id").distinct()

# Anti-join to get unmatched
axis_unmatched = axis_core.join(matched_axis_ids, on="axis_id", how="left_anti")
fin_unmatched = fin_core.join(matched_fin_ids, on="fin_id", how="left_anti")

# Ensure amounts are clean + enforce minimum amount threshold (v5.1)
axis_unmatched = axis_unmatched.filter(
    F.col(AXIS_AMOUNT_COL).isNotNull()
    & (F.col(AXIS_AMOUNT_COL) != 0)
    & (F.abs(F.col(AXIS_AMOUNT_COL)) >= MIN_GREEDY_AMOUNT)
)
fin_unmatched = fin_unmatched.filter(
    F.col(FIN_AMOUNT_COL).isNotNull()
    & (F.abs(F.col(FIN_AMOUNT_COL)) >= MIN_GREEDY_AMOUNT)
)

# ── SAP Account filter on Finstore pool (v5.6) ──────────────────
# Count BEFORE filter (materialises fin_unmatched — one-off cost)
_fin_before_sap = fin_unmatched.count()
_axis_before     = axis_unmatched.count()   # materialise axis pool too

if GREEDY_SAP_FILTER_ENABLED and GREEDY_SAP_FILTER_COL in fin_unmatched.columns:
    _sap_broadcast = F.broadcast(
        spark.createDataFrame(
            [(v,) for v in SAP_ACCOUNT_LIST],
            [GREEDY_SAP_FILTER_COL]
        )
    )
    fin_unmatched = fin_unmatched.join(
        _sap_broadcast,
        on=GREEDY_SAP_FILTER_COL,
        how="inner"
    )
    _fin_after_sap = fin_unmatched.count()
    _reduction_pct = (1 - _fin_after_sap / max(_fin_before_sap, 1)) * 100

    print(f"\n── SAP Account Filter (v5.6) ──")
    print(f"  Column           : {GREEDY_SAP_FILTER_COL}")
    print(f"  Accounts in list : {len(SAP_ACCOUNT_LIST)}")
    print(f"  Finstore BEFORE  : {_fin_before_sap:>12,}")
    print(f"  Finstore AFTER   : {_fin_after_sap:>12,}")
    print(f"  Removed          : {_fin_before_sap - _fin_after_sap:>12,}  ({_reduction_pct:.1f}% reduction)")
elif GREEDY_SAP_FILTER_ENABLED:
    print(f"\n⚠️  SAP filter enabled but column '{GREEDY_SAP_FILTER_COL}' not found in Finstore.")
    print(f"    Available columns: {fin_unmatched.columns}")
    print(f"    Proceeding with full Finstore pool ({_fin_before_sap:,} records).")
    _fin_after_sap = _fin_before_sap
else:
    print(f"\n  SAP filter disabled. Finstore pool: {_fin_before_sap:,} records.")
    _fin_after_sap = _fin_before_sap

# Normalise counterparty for join
axis_unmatched = axis_unmatched.withColumn(
    "cpty_str", F.trim(F.col("CounterpartyId").cast("string"))
)
fin_unmatched = fin_unmatched.withColumn(
    "cpty_str", F.trim(F.col("counterpartyid").cast("string"))
)

# Persist — materialises during greedy Strategy 1
axis_unmatched = axis_unmatched.persist(StorageLevel.MEMORY_AND_DISK)
fin_unmatched = fin_unmatched.persist(StorageLevel.MEMORY_AND_DISK)

print(f"\n{'='*80}")
print("LAYER 2: GREEDY / PROBABILISTIC MATCHING  (v5.6)")
print(f"{'='*80}")
print(f"RUN_GREEDY: {RUN_GREEDY}")
print(f"Min greedy amount: {MIN_GREEDY_AMOUNT:,.0f}")
print(f"Axis unmatched pool  : {_axis_before:,}")
print(f"Fin  unmatched pool  : {_fin_after_sap:,}  (was {_fin_before_sap:,} before SAP filter)")
print(f"S1 max iterations: {S1_MAX_ITER}  |  S2 max iterations: {S2_MAX_ITER}")
print(f"Lineage checkpoint every: {LINEAGE_CHECKPOINT_EVERY} iterations")
print("Unmatched pools prepared (will materialise during Strategy 1).")

---
### 🔹 Section 12: Greedy Strategy 1 — Amount + Counterparty (1%)

**Iterative shrinking-pool** approach (v5.4) — mirrors Pandas' `groupby → iterrows` with
running exclusion sets. Each iteration:
1. Join unmatched axis ↔ unmatched fin on `cpty_str` + tolerance filter
2. Pick best 1 fin per axis (min amount_diff)
3. Pick best 1 axis per fin (break ties)
4. Commit those matches → shrink both pools → repeat

v5.4 improvements:
- `S1_MAX_ITER` from config cell (reduced from 15 to 8)
- Lineage truncation via `localCheckpoint()` every `LINEAGE_CHECKPOINT_EVERY` iterations
- Individual batch DataFrames unpersisted after union
- Wrapped in try/except — BRD results survive greedy failure

In [ ]:
# ============================================================
# GREEDY STRATEGY 1: Amount + Counterparty — Iterative (v5.4)
# ============================================================
# Mirrors Pandas' sequential greedy with running exclusion sets.
#
# WHY ITERATIVE?  A single big join on cpty_str produces O(A×F) candidate
# pairs per counterparty group.  Popular counterparties (5K axis × 50K fin)
# create 250M+ rows *before* tolerance filtering.
#
# The iterative approach:
#   1. Join (small, because pools shrink each round)
#   2. Best-1 per axis (window, row_number)
#   3. Best-1 per fin  (window, row_number) → exclusive 1:1 batch
#   4. Commit batch → anti-join to shrink pools → repeat
#
# v5.4: Lineage truncation every LINEAGE_CHECKPOINT_EVERY iters,
#        unpersist batches after union, try/except resilience,
#        S1_MAX_ITER from config cell (was hardcoded 15)
# v5.3: Iterative shrinking-pool (replaces single-join + stable-marriage)
# v5.2: Removed SourceSystemName equi-join (Finstore has no system column)
# v5.1: MIN_GREEDY_AMOUNT already applied in unmatched pools

# Shared empty-result schema for both greedy strategies.
# Defined once here to avoid duplication and schema drift between S1 and S2.
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
_GREEDY_EMPTY_SCHEMA = StructType([
    StructField("axis_id",          StringType(),  True),
    StructField("fin_id",           StringType(),  True),
    StructField("amount_diff",      DoubleType(),  True),
    StructField("priority",         IntegerType(), True),
    StructField("category",         StringType(),  True),
    StructField("brd_priority",     IntegerType(), True),
    StructField("description",      StringType(),  True),
    StructField("amount_rel_diff",  DoubleType(),  True),
    StructField("key_strength",     IntegerType(), True),
    StructField("MatchLayer",       StringType(),  True),
    StructField("run_id",           StringType(),  True),
    StructField("batch_id",         StringType(),  True),
    StructField("rule_version",     StringType(),  True),
    StructField("match_timestamp",  StringType(),  True),
])

_greedy_failed = False  # flag for downstream resilience

if not RUN_GREEDY:
    print("⏭️  GREEDY SKIPPED (RUN_GREEDY=False)")
    greedy1_matches = spark.createDataFrame([], schema=_GREEDY_EMPTY_SCHEMA)
    greedy1_count = 0
else:
    print(f"\n--- Strategy 1: Amount + Counterparty ({GREEDY_TOLERANCE_PCT*100}% tolerance) [iterative, max {S1_MAX_ITER} iters] ---")

    try:
        greedy1_all_batches = []
        s1_axis_pool = axis_unmatched
        s1_fin_pool  = fin_unmatched

        for s1_iter in range(1, S1_MAX_ITER + 1):
            # 1. Join on counterparty (blocking key — matches Pandas groupby)
            candidates = (
                s1_axis_pool.alias("a")
                .join(
                    s1_fin_pool.alias("f"),
                    on=(F.col("a.cpty_str") == F.col("f.cpty_str")),
                    how="inner"
                )
                .filter(
                    (F.col("a.cpty_str").isNotNull())
                    & (F.col("a.cpty_str") != "")
                    & (F.col("a.cpty_str") != "nan")
                    & (F.col("a.cpty_str") != "None")
                )
                .withColumn("amount_diff", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}") - F.col(f"f.{FIN_AMOUNT_COL}")))
                .withColumn("tolerance",   F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")) * GREEDY_TOLERANCE_PCT)
                .filter(F.col("amount_diff") <= F.col("tolerance"))
                .select(
                    F.col("a.axis_id").alias("axis_id"),
                    F.col("f.fin_id").alias("fin_id"),
                    F.col("amount_diff"),
                    F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")).alias("axis_amount"),
                )
            )

            # 2. Best fin per axis (smallest amount_diff)
            w_axis = Window.partitionBy("axis_id").orderBy(F.col("amount_diff").asc(), F.col("fin_id").asc())
            best_per_axis = candidates.withColumn("_rn", F.row_number().over(w_axis)).filter(F.col("_rn") == 1).drop("_rn")

            # 3. Best axis per fin (break ties — exclusive 1:1)
            w_fin = Window.partitionBy("fin_id").orderBy(F.col("amount_diff").asc(), F.col("axis_id").asc())
            batch = best_per_axis.withColumn("_rn", F.row_number().over(w_fin)).filter(F.col("_rn") == 1).drop("_rn")

            batch_count = batch.count()
            print(f"  Iteration {s1_iter}: {batch_count:,} new exclusive pairs")

            if batch_count == 0:
                break

            # 4. Commit batch
            batch = batch.persist(StorageLevel.MEMORY_AND_DISK)
            greedy1_all_batches.append(batch)

            # 5. Shrink pools (anti-join matched IDs)
            matched_axis = batch.select("axis_id")
            matched_fin  = batch.select("fin_id")
            s1_axis_pool = s1_axis_pool.join(matched_axis, on="axis_id", how="left_anti")
            s1_fin_pool  = s1_fin_pool.join(matched_fin,  on="fin_id",  how="left_anti")

            # 6. Truncate lineage every N iterations to prevent catalyst explosion
            if s1_iter % LINEAGE_CHECKPOINT_EVERY == 0:
                s1_axis_pool = s1_axis_pool.localCheckpoint(eager=True)
                s1_fin_pool  = s1_fin_pool.localCheckpoint(eager=True)
                print(f"    ↳ lineage truncated (iteration {s1_iter})")

        # Combine all batches
        if greedy1_all_batches:
            greedy1_matches = greedy1_all_batches[0]
            for b in greedy1_all_batches[1:]:
                greedy1_matches = greedy1_matches.unionByName(b)
            # Add metadata columns
            greedy1_matches = (
                greedy1_matches
                .withColumn("priority",        F.lit(16).cast("int"))
                .withColumn("category",        F.lit("GREEDY"))
                .withColumn("brd_priority",    F.lit(1).cast("int"))
                .withColumn("description",     F.lit(f"Greedy: Amount+Counterparty ({GREEDY_TOLERANCE_PCT*100}%)"))
                .withColumn("amount_rel_diff", F.col("amount_diff") / F.greatest(F.col("axis_amount"), F.lit(1e-9)))
                .drop("axis_amount")
                .withColumn("key_strength",    F.lit(0).cast("int"))
                .withColumn("MatchLayer",      F.lit("GREEDY"))
                .withColumn("run_id",          F.lit(RUN_ID))
                .withColumn("batch_id",        F.lit(BATCH_ID))
                .withColumn("rule_version",    F.lit(RULE_VERSION))
                .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
            )
            # Unpersist individual batches — they're now part of the union
            for b in greedy1_all_batches:
                try:
                    b.unpersist()
                except Exception:
                    pass
        else:
            greedy1_matches = spark.createDataFrame([], schema=_GREEDY_EMPTY_SCHEMA)

        greedy1_matches = greedy1_matches.persist(StorageLevel.MEMORY_AND_DISK)
        greedy1_count = greedy1_matches.count()
        print(f"\nStrategy 1 total matches: {greedy1_count:,}")

    except Exception as e:
        print(f"\n❌ GREEDY S1 FAILED: {e}")
        print("   BRD results are safe (checkpoint saved). Creating empty greedy1.")
        _greedy_failed = True
        greedy1_matches = spark.createDataFrame([], schema=_GREEDY_EMPTY_SCHEMA)
        greedy1_matches = greedy1_matches.persist(StorageLevel.MEMORY_AND_DISK)
        greedy1_count = 0

---
### 🔹 Section 13: Greedy Strategy 2 — Amount Bucket (0.1%)

**Iterative shrinking-pool** (v5.4) — no counterparty requirement, just amount proximity.
Each iteration:
1. Bucket axis amounts (±1 bucket neighbour search)
2. Join on `search_bucket == amount_bucket` + strict 0.1% tolerance
3. Pick best 1:1 batch → shrink pools → repeat

v5.4: `S2_MAX_ITER` from config (reduced from 15 to 5), lineage truncation,
try/except resilience. v5.2: Removed SourceSystemName equi-join.

In [ ]:
# ============================================================
# GREEDY STRATEGY 2: Amount Bucket — Iterative (v5.4)
# ============================================================
# Same iterative shrinking-pool pattern as S1, but blocking on
# amount buckets (±1 neighbour) instead of counterparty.
# Stricter tolerance (0.1%) compensates for the broader join.
#
# v5.4: Lineage truncation, unpersist batches, try/except resilience,
#        S2_MAX_ITER from config (was hardcoded 15)
# v5.3: Iterative shrinking-pool (prevents O(A×F) bucket explosion)
# v5.2: Removed SourceSystemName equi-join
# v5.1: MIN_GREEDY_AMOUNT already applied in unmatched pools

if not RUN_GREEDY or _greedy_failed:
    if _greedy_failed:
        print("⏭️  GREEDY S2 SKIPPED (S1 failed)")
    else:
        print("⏭️  GREEDY S2 SKIPPED (RUN_GREEDY=False)")
    greedy2_matches = spark.createDataFrame([], schema=_GREEDY_EMPTY_SCHEMA)
    greedy2_matches = greedy2_matches.persist(StorageLevel.MEMORY_AND_DISK)
    greedy2_count = 0
else:
    print(f"\n--- Strategy 2: Amount Bucket ({STRICT_TOLERANCE_PCT*100}% tolerance) [iterative, max {S2_MAX_ITER} iters] ---")

    try:
        # Remove records already matched in Strategy 1 (both axis AND fin)
        greedy1_axis_ids = greedy1_matches.select("axis_id")
        greedy1_fin_ids  = greedy1_matches.select("fin_id")

        s2_axis_pool = axis_unmatched.join(greedy1_axis_ids, on="axis_id", how="left_anti")
        s2_fin_pool  = fin_unmatched.join(greedy1_fin_ids,  on="fin_id",  how="left_anti")

        # Pre-compute buckets on the base pools (before the loop)
        s2_axis_pool = s2_axis_pool.withColumn(
            "amount_bucket",
            (F.floor(F.col(AXIS_AMOUNT_COL) / BUCKET_SIZE) * BUCKET_SIZE).cast("long")
        )
        s2_fin_pool = s2_fin_pool.withColumn(
            "amount_bucket",
            (F.floor(F.col(FIN_AMOUNT_COL) / BUCKET_SIZE) * BUCKET_SIZE).cast("long")
        )

        # Broadcast tiny offset table (created once outside loop)
        bucket_offsets = spark.createDataFrame(
            [(-BUCKET_SIZE,), (0,), (BUCKET_SIZE,)],
            ["_offset"]
        )

        greedy2_all_batches = []

        for s2_iter in range(1, S2_MAX_ITER + 1):
            # 1. Expand axis buckets ±1
            axis_expanded = (
                s2_axis_pool
                .crossJoin(F.broadcast(bucket_offsets))
                .withColumn("search_bucket", F.col("amount_bucket") + F.col("_offset"))
                .drop("_offset")
            )

            # 2. Join on bucket + strict tolerance
            candidates = (
                axis_expanded.alias("a")
                .join(
                    s2_fin_pool.alias("f"),
                    on=(F.col("a.search_bucket") == F.col("f.amount_bucket")),
                    how="inner"
                )
                .withColumn("amount_diff", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}") - F.col(f"f.{FIN_AMOUNT_COL}")))
                .withColumn("tolerance",   F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")) * STRICT_TOLERANCE_PCT)
                .filter(F.col("amount_diff") <= F.col("tolerance"))
                .dropDuplicates(["axis_id", "fin_id"])
                .select(
                    F.col("a.axis_id").alias("axis_id"),
                    F.col("f.fin_id").alias("fin_id"),
                    F.col("amount_diff"),
                    F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")).alias("axis_amount"),
                )
            )

            # 3. Best fin per axis
            w_axis = Window.partitionBy("axis_id").orderBy(F.col("amount_diff").asc(), F.col("fin_id").asc())
            best_per_axis = candidates.withColumn("_rn", F.row_number().over(w_axis)).filter(F.col("_rn") == 1).drop("_rn")

            # 4. Best axis per fin (exclusive 1:1)
            w_fin = Window.partitionBy("fin_id").orderBy(F.col("amount_diff").asc(), F.col("axis_id").asc())
            batch = best_per_axis.withColumn("_rn", F.row_number().over(w_fin)).filter(F.col("_rn") == 1).drop("_rn")

            batch_count = batch.count()
            print(f"  Iteration {s2_iter}: {batch_count:,} new exclusive pairs")

            if batch_count == 0:
                break

            # 5. Commit batch
            batch = batch.persist(StorageLevel.MEMORY_AND_DISK)
            greedy2_all_batches.append(batch)

            # 6. Shrink pools
            matched_axis = batch.select("axis_id")
            matched_fin  = batch.select("fin_id")
            s2_axis_pool = s2_axis_pool.join(matched_axis, on="axis_id", how="left_anti")
            s2_fin_pool  = s2_fin_pool.join(matched_fin,  on="fin_id",  how="left_anti")

            # 7. Truncate lineage every N iterations
            if s2_iter % LINEAGE_CHECKPOINT_EVERY == 0:
                s2_axis_pool = s2_axis_pool.localCheckpoint(eager=True)
                s2_fin_pool  = s2_fin_pool.localCheckpoint(eager=True)
                print(f"    ↳ lineage truncated (iteration {s2_iter})")

        # Combine all batches
        if greedy2_all_batches:
            greedy2_matches = greedy2_all_batches[0]
            for b in greedy2_all_batches[1:]:
                greedy2_matches = greedy2_matches.unionByName(b)
            greedy2_matches = (
                greedy2_matches
                .withColumn("priority",        F.lit(17).cast("int"))
                .withColumn("category",        F.lit("GREEDY"))
                .withColumn("brd_priority",    F.lit(2).cast("int"))
                .withColumn("description",     F.lit(f"Greedy: Amount Bucket Strict ({STRICT_TOLERANCE_PCT*100}%)"))
                .withColumn("amount_rel_diff", F.col("amount_diff") / F.greatest(F.col("axis_amount"), F.lit(1e-9)))
                .drop("axis_amount")
                .withColumn("key_strength",    F.lit(0).cast("int"))
                .withColumn("MatchLayer",      F.lit("GREEDY"))
                .withColumn("run_id",          F.lit(RUN_ID))
                .withColumn("batch_id",        F.lit(BATCH_ID))
                .withColumn("rule_version",    F.lit(RULE_VERSION))
                .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
            )
            # Unpersist individual batches — they're now part of the union
            for b in greedy2_all_batches:
                try:
                    b.unpersist()
                except Exception:
                    pass
        else:
            greedy2_matches = spark.createDataFrame([], schema=_GREEDY_EMPTY_SCHEMA)

        greedy2_matches = greedy2_matches.persist(StorageLevel.MEMORY_AND_DISK)
        greedy2_count = greedy2_matches.count()
        print(f"\nStrategy 2 total matches: {greedy2_count:,}")

    except Exception as e:
        print(f"\n❌ GREEDY S2 FAILED: {e}")
        print("   BRD + S1 results are safe. Creating empty greedy2.")
        _greedy_failed = True
        greedy2_matches = spark.createDataFrame([], schema=_GREEDY_EMPTY_SCHEMA)
        greedy2_matches = greedy2_matches.persist(StorageLevel.MEMORY_AND_DISK)
        greedy2_count = 0

---
### 🔹 Section 14: Greedy Layer Summary

In [ ]:
# ============================================================
# LAYER 2 (GREEDY) SUMMARY  (v5.4)
# ============================================================
# ⚡ No new Spark jobs — uses pre-computed counts.

total_greedy = greedy1_count + greedy2_count

print(f"\n{'='*60}")
print("LAYER 2 (GREEDY) SUMMARY  (v5.4 — iterative shrinking-pool)")
print(f"{'='*60}")
if not RUN_GREEDY:
    print("  ⏭️  Greedy was SKIPPED (RUN_GREEDY=False)")
elif _greedy_failed:
    print("  ⚠️  Greedy PARTIALLY FAILED — partial results only")
print(f"Strategy 1 (Amount+Counterparty 1%): {greedy1_count:,}")
print(f"Strategy 2 (Amount Bucket 0.1%):     {greedy2_count:,}")
print(f"Total Greedy matches:                {total_greedy:,}")
print(f"Greedy rate:                         {total_greedy/ORIGINAL_AXIS_COUNT*100:.2f}%")

In [ ]:
# ============================================================
# GREEDY MATCH QUALITY DIAGNOSTICS  (v5.4)
# ============================================================
# These checks validate that greedy matches are genuine and not
# false positives from amount coincidence.
# v5.4: Guards all sections with total_greedy > 0 check.

print(f"\n{'='*80}")
print("GREEDY MATCH QUALITY DIAGNOSTICS")
print(f"{'='*80}")

if total_greedy == 0:
    print("  ⏭️  No greedy matches to diagnose (greedy skipped, failed, or found no matches).")
else:
    # ── 1. Fin_id exclusivity within greedy ──────────────────────────────────
    # Each fin_id must appear at most once across greedy S1 + S2 combined
    greedy_all = greedy1_matches.select("axis_id", "fin_id", "description").unionByName(
        greedy2_matches.select("axis_id", "fin_id", "description")
    )
    fin_reuse = (
        greedy_all
        .groupBy("fin_id")
        .agg(F.count("*").alias("cnt"))
        .filter(F.col("cnt") > 1)
    )
    fin_reuse_count = fin_reuse.count()
    if fin_reuse_count > 0:
        print(f"  ⚠️  {fin_reuse_count:,} fin_id(s) matched to MULTIPLE Axis trades in greedy")
        fin_reuse.orderBy(F.col("cnt").desc()).show(10, truncate=False)
    else:
        print(f"  ✅ Fin_id exclusivity: every fin_id matched at most once in greedy")

    # ── 2. Fin_id exclusivity across BRD + Greedy ───────────────────────────
    brd_fin_ids = brd_matches.select("fin_id").distinct()
    greedy_fin_ids = greedy_all.select("fin_id").distinct()
    cross_layer_fin_overlap = brd_fin_ids.intersect(greedy_fin_ids).count()
    if cross_layer_fin_overlap > 0:
        print(f"  ⚠️  {cross_layer_fin_overlap:,} fin_id(s) appear in BOTH BRD and Greedy layers")
    else:
        print(f"  ✅ No fin_id overlap between BRD and Greedy layers (pool anti-join working)")

    # ── 3. Amount difference distribution ────────────────────────────────────
    print(f"\n  Amount Difference Distribution (Greedy Strategy 1):")
    if greedy1_count > 0:
        greedy1_matches.select("amount_diff", "amount_rel_diff").summary(
            "min", "25%", "50%", "75%", "90%", "max"
        ).show(truncate=False)
    else:
        print("    (no Strategy 1 matches)")

    print(f"  Amount Difference Distribution (Greedy Strategy 2):")
    if greedy2_count > 0:
        greedy2_matches.select("amount_diff", "amount_rel_diff").summary(
            "min", "25%", "50%", "75%", "90%", "max"
        ).show(truncate=False)
    else:
        print("    (no Strategy 2 matches)")

    # ── 4. Greedy matches by SourceSystemName ────────────────────────────────
    print(f"\n  Greedy Matches by SourceSystemName:")
    greedy_with_system = (
        greedy1_matches.select("axis_id", F.lit("S1").alias("strategy"))
        .unionByName(greedy2_matches.select("axis_id", F.lit("S2").alias("strategy")))
        .join(axis_core.select("axis_id", "SourceSystemName"), on="axis_id", how="left")
    )
    greedy_with_system.groupBy("SourceSystemName", "strategy") \
        .count() \
        .orderBy(F.col("count").desc()) \
        .show(20, truncate=False)

print(f"{'='*80}")

---
## 🔹 Section 15: Final Results Consolidation

In [ ]:
# ============================================================
# FINAL CONSOLIDATION  —  v5.5
# ============================================================
# BRD layer: best-rule per axis, all fin matches within that rule
#   (1 axis can appear N times if its best rule hit N finstores)
# Greedy layers: exclusive 1:1 (iterative shrinking-pool v5.3+)
#   Each axis matches at most 1 fin, each fin matched by at most 1 axis.
#
# NO dropDuplicates on axis_id — within-rule many-to-many is intentional.
# Cross-rule duplication is already prevented by the best-priority filter.
#
# Pool removal uses DISTINCT axis_id / fin_id sets so
# greedy layers only process truly unmatched trades.
#
# v5.5: Added 1:1 traceability columns:
#   - match_rank:    ordinal rank of each finstore per axis_id
#                    (1 = best match by amount_diff, ties broken by fin_id)
#   - is_best_match: True for match_rank == 1  (the recommended 1:1 pair)
#   - match_type:    '1:1' when axis has exactly 1 finstore, 'multi_leg' otherwise
#   These columns let consumers instantly identify the single best
#   axis↔finstore mapping without needing custom logic.

# Union all match layers (lazy)
_all_matches_raw = (
    brd_matches
    .unionByName(greedy1_matches)
    .unionByName(greedy2_matches)
    # NO dropDuplicates — within-rule many-to-many rows are intentional
)

# ── 1:1 Traceability Columns (v5.5) ─────────────────────────────────────
# match_rank: rank finstore legs per axis, best (smallest) amount_diff first
_w_rank = Window.partitionBy("axis_id").orderBy(
    F.col("amount_diff").asc_nulls_last(),
    F.col("fin_id").asc(),               # deterministic tie-break
)
_all_matches_ranked = _all_matches_raw.withColumn(
    "match_rank", F.row_number().over(_w_rank)
)

# is_best_match: boolean flag — True only for the recommended 1:1 pair
_all_matches_ranked = _all_matches_ranked.withColumn(
    "is_best_match", F.col("match_rank") == 1
)

# match_type: '1:1' vs 'multi_leg' — based on how many finstores the axis has
_w_legs = Window.partitionBy("axis_id")
_all_matches_ranked = _all_matches_ranked.withColumn(
    "_n_legs", F.count("*").over(_w_legs)
)
_all_matches_ranked = _all_matches_ranked.withColumn(
    "match_type",
    F.when(F.col("_n_legs") == 1, F.lit("1:1")).otherwise(F.lit("multi_leg"))
).drop("_n_legs")

all_matches = _all_matches_ranked.persist(StorageLevel.MEMORY_AND_DISK)

# ── Count total rows and distinct axis IDs ───────────────────────────────
total_match_rows = all_matches.count()
total_unique_axis = all_matches.select("axis_id").distinct().count()
total_unique_fin  = all_matches.select("fin_id").distinct().count()

# Greedy counts (already 1 row per axis from resolve_matches)
total_greedy = greedy1_count + greedy2_count

# Unique axis matched = BRD unique + greedy (greedy axes are guaranteed
# disjoint from BRD axes thanks to anti-join pool removal)
total_matched = brd_unique_axis + total_greedy
final_match_rate = total_matched / ORIGINAL_AXIS_COUNT * 100

# Unmatched axis count
unmatched_axis_count = ORIGINAL_AXIS_COUNT - total_matched

# ── Cross-layer axis uniqueness assertion (greedy vs BRD) ────────────────
# Greedy axes must NOT overlap with BRD axes (pool anti-join ensures this).
greedy_axis = greedy1_matches.select("axis_id").union(greedy2_matches.select("axis_id"))
overlap = brd_matches.select("axis_id").intersect(greedy_axis)
overlap_count = overlap.count()

if overlap_count > 0:
    print(f"⚠️  WARNING: {overlap_count:,} axis_id(s) matched in BOTH BRD and Greedy layers!")
    print("    Investigate pool anti-join logic.")
else:
    print("✅ Cross-layer axis uniqueness check PASSED — no axis_id matched in both BRD and Greedy.")

# ── Verify total_unique_axis matches expected count ──────────────────────
if total_unique_axis != total_matched:
    print(f"⚠️  WARNING: distinct axis in all_matches ({total_unique_axis:,}) != "
          f"expected total_matched ({total_matched:,})")
    total_matched = total_unique_axis  # use actual safe count
else:
    print(f"✅ Unique axis count verified: {total_unique_axis:,}")

# Summary
many_to_many_rows = total_match_rows - total_unique_axis
print(f"ℹ️  Within-rule many-to-many: {many_to_many_rows:,} extra rows "
      f"({total_match_rows:,} total rows, {total_unique_axis:,} unique axis, "
      f"{total_unique_fin:,} unique fin)")

# ── 1:1 traceability summary (v5.5) ─────────────────────────────────────
best_match_count = all_matches.filter(F.col("is_best_match")).select("axis_id").distinct().count()
one_to_one_count = all_matches.filter(F.col("match_type") == "1:1").select("axis_id").distinct().count()
multi_leg_count  = all_matches.filter(F.col("match_type") == "multi_leg").select("axis_id").distinct().count()

print(f"\n── 1:1 Traceability (v5.5) ──")
print(f"  Axis with is_best_match=True: {best_match_count:,}  (should == total_unique_axis)")
print(f"  match_type='1:1':             {one_to_one_count:,}  ({one_to_one_count/max(total_unique_axis,1)*100:.1f}%)")
print(f"  match_type='multi_leg':       {multi_leg_count:,}  ({multi_leg_count/max(total_unique_axis,1)*100:.1f}%)")
print(f"  ℹ️  Filter on is_best_match=True for definitive 1 axis → 1 finstore mapping")

# Build unmatched sets using DISTINCT axis/fin IDs
all_matched_axis_ids = all_matches.select("axis_id").distinct()
all_matched_fin_ids  = all_matches.select("fin_id").distinct()

final_unmatched_axis = axis_core.join(all_matched_axis_ids, on="axis_id", how="left_anti")
final_unmatched_fin  = fin_core.join(all_matched_fin_ids, on="fin_id", how="left_anti")

print(f"\n{'='*80}")
print("FINAL RESULTS")
print(f"{'='*80}")
print(f"SCOPE: {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}")
print(f"{'-'*60}")
print(f"Total Axis (in-scope): {ORIGINAL_AXIS_COUNT:,}")
print(f"\nLayer 1 (BRD Rules):")
print(f"  {brd_unique_axis:,} unique axis matched, {brd_match_rows:,} match rows "
      f"(within-rule m:m) ({brd_match_rate:.2f}%)")
print(f"Layer 2 (Greedy):")
print(f"  {total_greedy:,} matched ({total_greedy/ORIGINAL_AXIS_COUNT*100:.2f}%)")
print(f"\nTOTAL UNIQUE AXIS MATCHED: {total_matched:,}")
print(f"TOTAL MATCH ROWS:          {total_match_rows:,}")
print(f"TOTAL UNMATCHED (Axis):    {ORIGINAL_AXIS_COUNT - total_matched:,}")
print(f"\n>>> FINAL MATCH RATE (unique axis): {total_matched / ORIGINAL_AXIS_COUNT * 100:.2f}% <<<")
print(f"\nℹ️  New columns: match_rank, is_best_match, match_type")
print(f"    Filter: is_best_match = True  →  definitive 1 axis → 1 finstore")
print(f"{'='*80}")

---
### 🔹 Section 15a: Match Count Verification (6 Invariants — Soft-Fail v5.4)

Six assertions validated before saving results. **v5.4: Soft-fail** — errors are
logged and `_verification_passed` is set to `False`, but the notebook does **not**
raise `AssertionError`. Results are still saved with a `verification_status` column
so downstream consumers can filter accordingly.

1. **Pool isolation**: no `axis_id` appears in both BRD and Greedy layers
2. **Partition**: no `axis_id` appears in both matched and unmatched sets
3. **Axis reconciliation**: `unique_matched_axis + unmatched_axis == ORIGINAL_AXIS_COUNT`
4. **Pair uniqueness**: no duplicate `(axis_id, fin_id)` pairs in all_matches
5. **Finstore reconciliation**: `unique_matched_fin + unmatched_fin == ORIGINAL_FINSTORE_COUNT`
6. **Greedy fin_id exclusivity**: each `fin_id` appears at most once across greedy S1+S2

> `axis_id` **can** appear multiple times within BRD matches if its best rule
> produced multiple finstore hits (within-rule many-to-many). The checks verify
> **unique axis** counts, not raw row counts.

In [ ]:
# ============================================================
# SECTION 15a: MATCH COUNT VERIFICATION  (v5.4)
# ============================================================
# Six invariants validated before saving.
# v5.4: SOFT-FAIL — logs errors and sets _verification_passed = False
#       but does NOT raise AssertionError.  Results are still saved
#       (downstream consumers check the flag / verification_status column).
#       This prevents losing hours of BRD + greedy work due to a single
#       invariant failure.
#
# CHECK 1: Pool isolation — no axis_id in both BRD and Greedy
# CHECK 2: Partition — no axis_id in both matched and unmatched
# CHECK 3: Axis reconciliation — unique_matched + unmatched == total
# CHECK 4: Pair uniqueness — no duplicate (axis_id, fin_id) in output
# CHECK 5: Finstore reconciliation — unique_matched_fin + unmatched_fin == total
# CHECK 6: Greedy fin_id exclusivity — each fin_id at most once in greedy

print(f"\n{'='*80}")
print("MATCH COUNT VERIFICATION  (6 invariants, v5.4 — soft-fail)")
print(f"{'='*80}")

errors = []

# ── CHECK 1: no axis_id in both BRD and Greedy (pool isolation) ──────
greedy_distinct_axis = (
    greedy1_matches.select("axis_id")
    .union(greedy2_matches.select("axis_id"))
    .distinct()
)
brd_greedy_overlap = brd_matches.select("axis_id").distinct().intersect(greedy_distinct_axis)
brd_greedy_overlap_count = brd_greedy_overlap.count()
if brd_greedy_overlap_count != 0:
    errors.append(
        f"CHECK 1 FAILED: {brd_greedy_overlap_count:,} axis_id(s) matched "
        f"in BOTH BRD and Greedy layers — pool anti-join is broken."
    )
else:
    print(f"✅ CHECK 1 PASSED: zero axis_id overlap between BRD and Greedy layers")

# ── CHECK 2: no axis_id in both matched and unmatched ────────────────
overlap_axis = all_matches.select("axis_id").distinct().intersect(final_unmatched_axis.select("axis_id"))
overlap_count = overlap_axis.count()
if overlap_count != 0:
    errors.append(
        f"CHECK 2 FAILED: {overlap_count:,} axis_id(s) appear in BOTH "
        f"matched and unmatched sets"
    )
else:
    print(f"✅ CHECK 2 PASSED: zero axis_id overlap between matched and unmatched")

# ── CHECK 3: Axis reconciliation identity ────────────────────────────
unmatched_axis_actual = final_unmatched_axis.count()
total_accounted = total_matched + unmatched_axis_actual
if total_accounted != ORIGINAL_AXIS_COUNT:
    errors.append(
        f"CHECK 3 FAILED: unique_matched_axis({total_matched:,}) + "
        f"unmatched_axis({unmatched_axis_actual:,}) = {total_accounted:,} "
        f"!= ORIGINAL_AXIS_COUNT({ORIGINAL_AXIS_COUNT:,})"
    )
else:
    print(f"✅ CHECK 3 PASSED: unique_matched_axis({total_matched:,}) + "
          f"unmatched_axis({unmatched_axis_actual:,}) == "
          f"ORIGINAL_AXIS_COUNT({ORIGINAL_AXIS_COUNT:,})")

# ── CHECK 4: Pair uniqueness — no duplicate (axis_id, fin_id) ────────
pair_total = total_match_rows
pair_distinct = all_matches.select("axis_id", "fin_id").distinct().count()
pair_dupes = pair_total - pair_distinct
if pair_dupes != 0:
    errors.append(
        f"CHECK 4 FAILED: {pair_dupes:,} duplicate (axis_id, fin_id) pairs "
        f"in all_matches ({pair_total:,} rows, {pair_distinct:,} distinct pairs)"
    )
else:
    print(f"✅ CHECK 4 PASSED: zero duplicate (axis_id, fin_id) pairs "
          f"({pair_total:,} rows == {pair_distinct:,} distinct pairs)")

# ── CHECK 5: Finstore reconciliation identity ────────────────────────
unmatched_fin_actual = final_unmatched_fin.count()
fin_accounted = total_unique_fin + unmatched_fin_actual
if fin_accounted != ORIGINAL_FINSTORE_COUNT:
    errors.append(
        f"CHECK 5 FAILED: unique_matched_fin({total_unique_fin:,}) + "
        f"unmatched_fin({unmatched_fin_actual:,}) = {fin_accounted:,} "
        f"!= ORIGINAL_FINSTORE_COUNT({ORIGINAL_FINSTORE_COUNT:,})"
    )
else:
    print(f"✅ CHECK 5 PASSED: unique_matched_fin({total_unique_fin:,}) + "
          f"unmatched_fin({unmatched_fin_actual:,}) == "
          f"ORIGINAL_FINSTORE_COUNT({ORIGINAL_FINSTORE_COUNT:,})")

# ── CHECK 6: Greedy fin_id exclusivity ───────────────────────────────
greedy_combined_fins = (
    greedy1_matches.select("fin_id")
    .unionAll(greedy2_matches.select("fin_id"))
)
greedy_fin_total = greedy_combined_fins.count()
greedy_fin_distinct = greedy_combined_fins.distinct().count()
greedy_fin_dupes = greedy_fin_total - greedy_fin_distinct
if greedy_fin_dupes != 0:
    errors.append(
        f"CHECK 6 FAILED: {greedy_fin_dupes:,} fin_id(s) reused across "
        f"greedy matches ({greedy_fin_total:,} total, {greedy_fin_distinct:,} distinct)"
    )
else:
    print(f"✅ CHECK 6 PASSED: zero fin_id reuse in greedy layer "
          f"({greedy_fin_total:,} total == {greedy_fin_distinct:,} distinct)")

# ── SOFT-FAIL: log errors but DO NOT raise ───────────────────────────
_verification_passed = len(errors) == 0

if errors:
    print(f"\n{'!'*80}")
    for e in errors:
        print(f"  ❌ {e}")
    print(f"{'!'*80}")
    print(f"\n⚠️  {len(errors)} verification check(s) failed.")
    print("   Results will STILL be saved (with verification_status='FAILED').")
    print("   Review failures before using downstream reports.")
    if _greedy_failed:
        print("   ℹ️  Greedy failure may explain some invariant violations.")
else:
    print(f"\n{'='*80}")
    print("ALL 6 VERIFICATION CHECKS PASSED ✅")
    print(f"{'='*80}")
    print(f"\n  Axis reconciliation:    {total_matched:,} matched + "
          f"{unmatched_axis_actual:,} unmatched = {ORIGINAL_AXIS_COUNT:,}")
    print(f"  Finstore reconciliation:{total_unique_fin:,} matched + "
          f"{unmatched_fin_actual:,} unmatched = {ORIGINAL_FINSTORE_COUNT:,}")
    print(f"  Pair uniqueness:        {pair_total:,} rows, all distinct (axis_id, fin_id)")
    print(f"  Greedy fin exclusivity: {greedy_fin_total:,} greedy fins, all distinct")
    print(f"  Multi-leg rows:         {pair_total - total_unique_axis:,} "
          f"(1-axis → N-distinct-fin, legitimate)")
    print(f"{'='*80}")

---
## 🔹 Section 15b: Save Base Matches (Narrow — Before Enrichment)

Save the core match results **before** the expensive wide-column enrichment join.
This provides a lightweight Delta table (~15 columns) for quick analysis,
while the enriched version with 100+ columns is saved separately after.

In [ ]:
# ============================================================
# SAVE BASE MATCHES — narrow schema, before enrichment (v5.5)
# ============================================================
# Lightweight output (~18 cols) for quick downstream queries.
# v5.5: Adds match_rank, is_best_match, match_type for 1:1 traceability.
# v5.4: Adds verification_status + greedy_status columns so
#        consumers know if results are fully verified.
#        Saves proceed even when verification or greedy failed.

_v_status = "PASSED" if _verification_passed else "FAILED"
_g_status = "SKIPPED" if not RUN_GREEDY else ("FAILED" if _greedy_failed else "OK")

print(f"Saving base (narrow) match results to: {OUTPUT_DIR}")
print(f"  verification_status: {_v_status}")
print(f"  greedy_status:       {_g_status}")
print(f"{'-'*60}")

# Tag all_matches with status columns before save
_all_matches_save = (
    all_matches
    .withColumn("verification_status", F.lit(_v_status))
    .withColumn("greedy_status", F.lit(_g_status))
)

# --- All matches (BRD + Greedy) — narrow ---
_all_matches_save.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_all_base")
print(f"✅ Saved Delta: matched_all_base ({total_match_rows:,} rows, {len(_all_matches_save.columns)} cols)")

# --- BRD only (narrow) ---
brd_matches.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_brd_layer")
print(f"✅ Saved Delta: matched_brd_layer ({brd_match_rows:,} rows)")

# --- Greedy only (narrow) ---
if total_greedy > 0:
    greedy_all_df = all_matches.filter(F.col("MatchLayer") == "GREEDY")
    greedy_all_df.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_greedy_layer")
    print(f"✅ Saved Delta: matched_greedy_layer ({total_greedy:,} rows)")
else:
    # Save empty greedy layer with correct schema
    spark.createDataFrame([], schema=_GREEDY_EMPTY_SCHEMA) \
        .write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_greedy_layer")
    print(f"✅ Saved Delta: matched_greedy_layer (0 rows — greedy {'skipped' if not RUN_GREEDY else 'empty/failed'})")

# --- Unmatched Axis (narrow core only) ---
final_unmatched_axis.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_axis_base")
print(f"✅ Saved Delta: unmatched_axis_base")

# --- Unmatched Finstore (narrow core only) ---
final_unmatched_fin.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_finstore_base")
print(f"✅ Saved Delta: unmatched_finstore_base")

print(f"\n{'='*60}")
print("BASE (NARROW) RESULTS SAVED — enrichment follows in Section 16.")
print(f"{'='*60}")

---
## 🔹 Section 16: Enrichment — Join Back Wide Columns

Now that matching is done on narrow core tables,
we join back the full 100+ columns for the final output.

In [ ]:
# ============================================================
# ENRICHMENT — Join back wide columns (100+ cols)
# ============================================================
# This is the ONLY place where wide DataFrames are joined.
# All matching was done on narrow core tables.
# ⚡ PERF: No .count() — these are written directly to Delta.
#    Counts come from the narrow base already computed.

# Rename wide columns to avoid collisions
axis_wide_renamed = axis_wide
for c in axis_wide.columns:
    if c != "axis_id":
        axis_wide_renamed = axis_wide_renamed.withColumnRenamed(c, f"{c}_Axis")

fin_wide_renamed = fin_wide
for c in fin_wide.columns:
    if c != "fin_id":
        fin_wide_renamed = fin_wide_renamed.withColumnRenamed(c, f"{c}_Finstore")

# Enrich matches
matches_enriched = (
    all_matches
    .join(axis_wide_renamed, on="axis_id", how="left")
    .join(fin_wide_renamed, on="fin_id", how="left")
)

# Add Variance column
axis_amt_col_enriched = f"{AXIS_AMOUNT_COL}_Axis"
fin_amt_col_enriched = f"{FIN_AMOUNT_COL}_Finstore"

matches_enriched = matches_enriched.withColumn(
    "Variance",
    F.col(axis_amt_col_enriched) - F.col(fin_amt_col_enriched)
)

# Enrich unmatched axis
unmatched_axis_enriched = (
    final_unmatched_axis
    .select("axis_id")
    .join(axis_wide_renamed, on="axis_id", how="left")
)

# Enrich unmatched finstore
unmatched_fin_enriched = (
    final_unmatched_fin
    .select("fin_id")
    .join(fin_wide_renamed, on="fin_id", how="left")
)

# ⚡ No .count() here — enriched DFs are written directly to Delta in Section 19.
print(f"Enriched DataFrames prepared (lazy).")
print(f"  matches_enriched columns: {len(matches_enriched.columns)}")
print(f"  unmatched_axis_enriched columns: {len(unmatched_axis_enriched.columns)}")
print(f"  unmatched_fin_enriched columns: {len(unmatched_fin_enriched.columns)}")
print("Will materialise during Delta write in Section 19.")

---
## 🔹 Section 17: Matches by System Breakdown

In [ ]:
# ============================================================
# BREAKDOWN: Matches by Source System
# ============================================================

print(f"\n{'='*60}")
print("MATCHES BY SOURCE SYSTEM")
print(f"{'='*60}")

# BRD matches by system
print("\nBRD Matches by system:")
brd_by_system = (
    brd_matches
    .join(axis_core.select("axis_id", "SourceSystemName"), on="axis_id", how="left")
    .groupBy("SourceSystemName")
    .count()
    .orderBy(F.desc("count"))
)
brd_by_system.show(20, truncate=False)

# Greedy matches by system
if total_greedy > 0:
    print("Greedy Matches by system:")
    greedy_all = greedy1_matches.unionByName(greedy2_matches)
    greedy_by_system = (
        greedy_all
        .join(axis_core.select("axis_id", "SourceSystemName"), on="axis_id", how="left")
        .groupBy("SourceSystemName")
        .count()
        .orderBy(F.desc("count"))
    )
    greedy_by_system.show(20, truncate=False)
else:
    print("Greedy Matches by system: (no greedy matches)")

---
## 🔹 Section 18: Remaining Unmatched by System

In [ ]:
# ============================================================
# UNMATCHED BY SYSTEM
# ============================================================

print(f"\n{'='*60}")
print("REMAINING UNMATCHED BY SYSTEM")
print(f"{'='*60}")

unmatched_by_system = (
    final_unmatched_axis
    .groupBy("SourceSystemName")
    .count()
    .orderBy(F.desc("count"))
)
unmatched_by_system.show(20, truncate=False)

---
## 🔹 Section 18b: Data Quality Validation (Best Practice §7)

Validate core column data contracts before outputs are consumed downstream.

In [ ]:
# ============================================================
# DATA QUALITY VALIDATION  (Best Practice §7)
# ============================================================
# ⚡ PERF: Instead of N separate .count() calls (was 14 Spark jobs!),
#    compute all violation counts in a SINGLE aggregation per DataFrame.

def validate_dataframe_fast(df, name, checks):
    """Run all checks in a single aggregation pass — 1 Spark job per DF."""
    agg_exprs = [
        F.sum(F.when(expr, 1).otherwise(0)).alias(desc)
        for desc, expr in checks
    ]
    result_row = df.agg(*agg_exprs).collect()[0]
    rows = [(name, desc, int(result_row[desc])) for desc, _ in checks]
    return spark.createDataFrame(rows, ["dataset", "check", "violation_count"])

# --- Axis Core checks (1 Spark job instead of 5) ---
axis_checks = [
    ("null_axis_id",            F.col("axis_id").isNull()),
    ("null_CounterpartyId",     F.col("CounterpartyId").isNull()),
    ("null_NotionalAmount",     F.col(AXIS_AMOUNT_COL).isNull()),
    ("negative_Notional",       F.col(AXIS_AMOUNT_COL) < 0),
    ("null_ReconSubProduct",    F.col("ReconSubProduct").isNull()),
]

# --- Finstore Core checks (1 Spark job instead of 4) ---
fin_checks = [
    ("null_fin_id",             F.col("fin_id").isNull()),
    ("null_counterpartyid",     F.col("counterpartyid").isNull()),
    ("null_NotionalAmount",     F.col(FIN_AMOUNT_COL).isNull()),
    ("negative_Notional",       F.col(FIN_AMOUNT_COL) < 0),
]

# --- Match output checks (1 Spark job instead of 5) ---
match_checks = [
    ("null_axis_id",        F.col("axis_id").isNull()),
    ("null_fin_id",         F.col("fin_id").isNull()),
    ("null_description",    F.col("description").isNull()),
    ("null_run_id",         F.col("run_id").isNull()),
    ("amount_rel_diff_gt_1", F.col("amount_rel_diff") > 1.0),
]

dq_axis    = validate_dataframe_fast(axis_core,   "axis_core",   axis_checks)
dq_fin     = validate_dataframe_fast(fin_core,    "fin_core",    fin_checks)
dq_matches = validate_dataframe_fast(all_matches, "all_matches", match_checks)

dq_report = dq_axis.union(dq_fin).union(dq_matches)
print("=== Data Quality Validation (3 Spark jobs instead of 14) ===")
dq_report.show(50, truncate=False)

violations = dq_report.filter(F.col("violation_count") > 0).count()
if violations > 0:
    print(f"⚠️  {violations} check(s) have violations — review before promoting to Gold.")
else:
    print("✅ All data quality checks passed.")

---
## 🔹 Section 18c: Explainability — Unmatched Reason Breakdown (Best Practice §6)

Every unmatched trade should carry a *reason* so downstream consumers can investigate.

In [ ]:
# ============================================================
# EXPLAINABILITY — UNMATCHED REASON BREAKDOWN  (Best Practice §6)
# Classify why each unmatched trade failed to match.
# ============================================================

# --- Axis unmatched reasons ---
# Check if the trade appeared as a candidate in ANY rule but was outcompeted
axis_candidates_ever = candidates_layer1.select("axis_id").distinct()

axis_unmatched_reasons = (
    final_unmatched_axis
    .join(axis_candidates_ever, on="axis_id", how="left")
    .withColumn(
        "unmatched_reason",
        F.when(axis_candidates_ever["axis_id"].isNotNull(),
               F.lit("candidate_existed_but_consumed_by_higher_priority"))
         .otherwise(F.lit("no_candidate_key_found"))
    )
    .select("axis_id", "SourceSystemTradeId", "CounterpartyId", "ReconSubProduct",
            AXIS_AMOUNT_COL, "unmatched_reason")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
)

# --- Finstore unmatched reasons ---
fin_candidates_ever = candidates_layer1.select("fin_id").distinct()

fin_unmatched_reasons = (
    final_unmatched_fin
    .join(fin_candidates_ever, on="fin_id", how="left")
    .withColumn(
        "unmatched_reason",
        F.when(fin_candidates_ever["fin_id"].isNotNull(),
               F.lit("candidate_existed_but_consumed_by_higher_priority"))
         .otherwise(F.lit("no_candidate_key_found"))
    )
    .select("fin_id", "tradeid", "counterpartyid",
            FIN_AMOUNT_COL, "unmatched_reason")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
)

# --- Summary ---
print("=== Unmatched Axis Reason Breakdown ===")
axis_unmatched_reasons.groupBy("unmatched_reason").count().show(truncate=False)

print("=== Unmatched Finstore Reason Breakdown ===")
fin_unmatched_reasons.groupBy("unmatched_reason").count().show(truncate=False)

# Optionally persist for audit
# axis_unmatched_reasons.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_axis_reasons")
# fin_unmatched_reasons.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_fin_reasons")

---
## 🔹 Section 19: Save Results

**Best practice:** Save as Delta for downstream consumption;
export CSV only for legacy consumers / samples.

In [ ]:
# ============================================================
# SAVE ENRICHED RESULTS — Delta (wide tables with 100+ columns)
# ============================================================
# Base (narrow) results were already saved in Section 15b.
# This section saves the enriched (wide) versions for full reporting.
# ⚡ PERF: No .count() — write directly. Counts use previously computed values.

print(f"Saving enriched (wide) results to: {OUTPUT_DIR}")
print(f"{'-'*60}")

# All matches enriched (wide — full 100+ columns)
matches_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_all_enriched")
print(f"✅ Saved Delta: matched_all_enriched ({total_matched:,} rows)")

# Unmatched Axis (wide)
unmatched_axis_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_axis_enriched")
print(f"✅ Saved Delta: unmatched_axis_enriched")

# Unmatched Finstore (wide)
unmatched_fin_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_finstore_enriched")
print(f"✅ Saved Delta: unmatched_finstore_enriched")

# ---- OPTIMIZE + ZORDER (Best Practice §5 — Partitioning & Layout) ----
# With delta.optimizeWrite + delta.autoCompact enabled in Section 1,
# files are already well-sized. ZORDER further optimises read patterns.
# Uncomment on Databricks:
#
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/matched_all_enriched` ZORDER BY (axis_id, fin_id, priority)")
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/matched_all_base` ZORDER BY (axis_id, priority)")
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/unmatched_axis_enriched` ZORDER BY (axis_id)")
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/unmatched_finstore_enriched` ZORDER BY (fin_id)")
# print("Delta tables OPTIMIZED + ZORDERed.")

print("\nAll enriched results saved successfully.")

---
## 🔹 Section 20: Summary Report

In [ ]:
# ============================================================
# SUMMARY REPORT (v5.0)
# ============================================================
# ⚡ PERF: Uses only pre-computed counts — no new Spark jobs.

report = f"""
{'='*80}
DERIVATIVES MATCHING — COMPLETE RESULTS (BRD + GREEDY)
PySpark / Databricks Production — v5.4
{'='*80}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Run ID:    {RUN_ID}
Batch ID:  {BATCH_ID}
Rule Ver:  {RULE_VERSION}
Scope:     {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}
Greedy:    {_g_status}
Verified:  {_v_status}

{'='*80}
INPUT DATA
{'='*80}
Total Axis (original):  {ORIGINAL_AXIS_COUNT_FULL:,}
Total Finstore:         {ORIGINAL_FINSTORE_COUNT:,}
Excluded (SOPHIS/DELTA1): {excluded_count:,}
Axis (in-scope):        {ORIGINAL_AXIS_COUNT:,}

{'='*80}
LAYER 1: BRD DETERMINISTIC MATCHING (Sequential Waterfall v5.0)
{'='*80}
Rules executed: 15 (3 SOPHIS + 10 OTC + 2 ETD)
Architecture: safe_rule_join() with sequential pool removal
BRD match rows (m:m): {brd_match_rows:,}
Unique Axis matched: {brd_unique_axis:,}
Unique Finstore used: {brd_unique_fin:,}
Match Rate (unique axis): {brd_match_rate:.2f}%

{'='*80}
LAYER 2: GREEDY/PROBABILISTIC MATCHING
{'='*80}
Strategy 1 (Amount+Counterparty, {GREEDY_TOLERANCE_PCT*100:.1f}%): {greedy1_count:,}
Strategy 2 (Amount Strict, {STRICT_TOLERANCE_PCT*100:.1f}%): {greedy2_count:,}
Total Greedy: {total_greedy:,}

{'='*80}
COMBINED RESULTS
{'='*80}
Total Unique Axis Matched: {total_matched:,}
Total Match Rows:    {total_match_rows:,}
Total Unmatched:     {unmatched_axis_count:,}
Final Match Rate (unique axis): {final_match_rate:.2f}%

{'='*80}
OUTPUT TABLES
{'='*80}
- matched_all_base       (narrow — match metadata, {total_match_rows:,} rows)
- matched_brd_layer      (narrow — BRD only, {brd_match_rows:,} rows)
- matched_greedy_layer   (narrow — Greedy only, {total_greedy:,} rows)
- matched_all_enriched   (wide — 100+ cols, {total_match_rows:,} rows)
- unmatched_axis_base    (narrow — core cols)
- unmatched_axis_enriched(wide — all cols)
- unmatched_finstore_base(narrow — core cols)
- unmatched_finstore_enriched (wide — all cols)

{'='*80}
v5.4 GUARDRAILS & RESILIENCE
{'='*80}
- Sequential waterfall: each rule only sees unmatched Axis trades
- safe_rule_join wrapper: key cleaning, pre-join diagnostics, circuit breaker
- Per-key Finstore cap: MAX_FIN_PER_KEY = {MAX_FIN_PER_KEY}
- Top-K per axis per rule: DEFAULT_TOP_K = {DEFAULT_TOP_K}
- BRD checkpoint saved before greedy (SAVE_BRD_CHECKPOINT={SAVE_BRD_CHECKPOINT})
- Greedy wrapped in try/except — BRD results survive greedy failure
- Verification soft-fail — results saved even if checks fail
- Lineage truncation every {LINEAGE_CHECKPOINT_EVERY} iterations in greedy loops

{'='*80}
PERFORMANCE OPTIMISATIONS
{'='*80}
- Core/Wide split: matching on {len(AXIS_CORE_COLS)} Axis + {len(FIN_CORE_COLS)} Finstore columns
- Sequential waterfall with accumulated anti-join (true Pandas parity)
- Iterative shrinking-pool greedy (S1 max {S1_MAX_ITER}, S2 max {S2_MAX_ITER} iters)
- Blocked joins for greedy (counterparty / amount buckets)
- Native Spark SQL functions (no Python UDFs)
- AQE + skew join + local shuffle reader enabled
- Shuffle partitions: 320 (tuned for 160-core cluster)
- Broadcast threshold: 256 MB
- Delta optimizeWrite + autoCompact enabled
- MEMORY_AND_DISK persist for scale safety

{'='*80}
DATA QUALITY (Best Practice §7)
{'='*80}
- Core column null/range checks executed (single-pass aggregation)
{'='*80}
"""

print(report)

try:
    dbutils.fs.put(f"{OUTPUT_DIR}/matching_summary_report.txt", report, overwrite=True)
    print(f"Summary report saved via dbutils to: {OUTPUT_DIR}/matching_summary_report.txt")
except NameError:
    import os
    report_path = OUTPUT_DIR.replace("dbfs:", "/dbfs") if OUTPUT_DIR.startswith("dbfs:") else OUTPUT_DIR
    os.makedirs(report_path, exist_ok=True)
    with open(os.path.join(report_path, "matching_summary_report.txt"), "w") as f:
        f.write(report)
    print(f"Summary report saved locally to: {report_path}/matching_summary_report.txt")

---
## 🔹 Section 21: Cleanup — Unpersist Cached DataFrames

In [ ]:
# ============================================================
# CLEANUP — release persisted DataFrames
# ============================================================

for df_to_unpersist in [axis_core, fin_core, candidates_layer1,
                         brd_matches, axis_unmatched, fin_unmatched,
                         greedy1_matches, greedy2_matches, all_matches]:
    try:
        df_to_unpersist.unpersist()
    except Exception:
        pass

print("All persisted DataFrames unpersisted. Pipeline complete.")

---
## 🔹 Section 22: Accuracy Report

Detailed breakdown of match results across all layers, rules, and strategies.
Mirrors the summary report from the Pandas reference notebook so results
can be compared directly between implementations.

In [ ]:
# ============================================================
# SECTION 22: ACCURACY REPORT
# ============================================================
# Mirrors the Pandas notebook summary structure so outputs
# can be compared directly between implementations.
# All variables come from prior cells — no new Spark jobs.

from datetime import datetime

# ── Per-rule breakdown from the already-cached brd_matches ───────────────
# One single Spark job to get counts per rule description.
rule_breakdown = (
    brd_matches
    .groupBy("priority", "category", "description")
    .count()
    .orderBy("priority")
    .collect()
)

# ── Match rate calculations (based on UNIQUE axis, not rows) ────────────
brd_match_rate_report    = brd_unique_axis / ORIGINAL_AXIS_COUNT * 100
greedy_match_rate_report = total_greedy    / ORIGINAL_AXIS_COUNT * 100
final_match_rate_report  = total_matched   / ORIGINAL_AXIS_COUNT * 100
unmatched_count_report   = ORIGINAL_AXIS_COUNT - total_matched

# ── Build report string ──────────────────────────────────────────────────
rule_lines = "\n".join(
    f"  [{row['category']:6s} P{row['priority']:02d}] {row['description'][:55]:<55s} : {row['count']:>8,}"
    for row in rule_breakdown
)

report = f"""
{'='*80}
DERIVATIVES MATCHING — ACCURACY REPORT  (BRD + GREEDY)
PySpark / Databricks  |  Run: {RUN_ID[:8]}...  |  Batch: {BATCH_ID}
{'='*80}

Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Rule Ver  : {RULE_VERSION}
Scope     : {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}

{'='*80}
INPUT DATA
{'='*80}
  Total Axis (original) : {ORIGINAL_AXIS_COUNT_FULL:>12,}
  Total Finstore        : {ORIGINAL_FINSTORE_COUNT:>12,}
  Excluded (scope)      : {excluded_count:>12,}
  Axis (in-scope)       : {ORIGINAL_AXIS_COUNT:>12,}

{'='*80}
LAYER 1: BRD DETERMINISTIC MATCHING  (15 rules: 3 SOPHIS + 10 OTC + 2 ETD)
{'='*80}
{rule_lines}
{'─'*80}
  BRD match rows (w/r m:m): {brd_match_rows:>12,}
  Unique Axis matched   : {brd_unique_axis:>12,}
  Unique Finstore used  : {brd_unique_fin:>12,}
  BRD Match Rate (axis) : {brd_match_rate_report:>11.2f}%

{'='*80}
LAYER 2: GREEDY / PROBABILISTIC MATCHING
{'='*80}
  Strategy 1  Amount + Counterparty ({GREEDY_TOLERANCE_PCT*100:.1f}%) : {greedy1_count:>8,}
  Strategy 2  Amount Strict         ({STRICT_TOLERANCE_PCT*100:.1f}%) : {greedy2_count:>8,}
{'─'*80}
  Total Greedy matched  : {total_greedy:>12,}
  Greedy Match Rate     : {greedy_match_rate_report:>11.2f}%

{'='*80}
FINAL COMBINED RESULTS
{'='*80}
  BRD (unique axis)     : {brd_unique_axis:>12,}  ({brd_match_rate_report:.2f}%)
  Greedy                : {total_greedy:>12,}  ({greedy_match_rate_report:.2f}%)
{'─'*80}
  TOTAL UNIQUE MATCHED  : {total_matched:>12,}
  TOTAL MATCH ROWS      : {total_match_rows:>12,}
  TOTAL UNMATCHED (Axis): {unmatched_count_report:>12,}

  >>> FINAL MATCH RATE (unique axis): {final_match_rate_report:>10.2f}% <<<

{'='*80}
OUTPUT TABLES  (saved to {OUTPUT_DIR})
{'='*80}
  Narrow (base — before enrichment):
    - matched_all_base         ({total_match_rows:,} rows, ~15 cols)
    - matched_brd_layer        ({brd_match_rows:,} rows)
    - matched_greedy_layer     ({total_greedy:,} rows)
    - unmatched_axis_base
    - unmatched_finstore_base

  Wide (enriched — 100+ cols):
    - matched_all_enriched     ({total_match_rows:,} rows)
    - unmatched_axis_enriched
    - unmatched_finstore_enriched

{'='*80}
MATCHING SEMANTICS  (v5.4 — Sequential Waterfall + Iterative Greedy)
{'='*80}
  BRD: safe_rule_join() sequential execution with pool removal : Pandas waterfall
  Each rule only sees unmatched Axis (anti-join pool removal)  : exact Pandas parity
  Within-rule many-to-many preserved (multi-leg instruments)   : pd.merge(inner)
  Greedy layers: iterative shrinking-pool (v5.3+)             : Pandas set tracking
    - Same fin_id cannot be matched by multiple Axis trades
    - No SourceSystemName equi-join (removed v5.2 — Finstore has no system col)
    - MIN_GREEDY_AMOUNT={MIN_GREEDY_AMOUNT} prevents near-zero false positives
    - Lineage truncation every {LINEAGE_CHECKPOINT_EVERY} iterations (v5.4)
  Pool removal uses DISTINCT axis/fin IDs                      : Pandas set.unique
  Match rate based on UNIQUE axis count                        : Pandas .nunique()
  6 runtime assertions verified in Section 15a (soft-fail v5.4)
  BRD checkpoint saved before greedy (SAVE_BRD_CHECKPOINT={SAVE_BRD_CHECKPOINT})
  Greedy status: {_g_status}  |  Verification: {_v_status}

{'='*80}
"""

print(report)

# ── Save report ──────────────────────────────────────────────────────────
try:
    # Databricks
    dbutils.fs.put(f"{OUTPUT_DIR}/accuracy_report.txt", report, overwrite=True)
    print(f"✅ Accuracy report saved via dbutils to: {OUTPUT_DIR}/accuracy_report.txt")
except NameError:
    # Local / non-Databricks fallback
    import os
    report_path = OUTPUT_DIR.replace("dbfs:", "/dbfs") if OUTPUT_DIR.startswith("dbfs:") else OUTPUT_DIR
    os.makedirs(report_path, exist_ok=True)
    out_file = os.path.join(report_path, "accuracy_report.txt")
    with open(out_file, "w") as f:
        f.write(report)
    print(f"✅ Accuracy report saved locally to: {out_file}")

---
## 🔹 Section 22b: POC Summary Report Builder (v5.1)

Comprehensive production-grade summary combining:
- Per-rule metrics with waterfall pool removal stats
- Match quality indicators (amount variance, 1-to-N distribution)
- P15 explosion diagnosis with before/after comparison
- Sequential waterfall effectiveness summary

In [ ]:
# ============================================================
# SECTION 22b: POC SUMMARY REPORT BUILDER (v5.1)
# ============================================================
# Comprehensive production-grade summary combining rule diagnostics,
# match quality metrics, and sequential waterfall effectiveness.
# All data comes from _rule_diagnostics + pre-computed counts — minimal new Spark jobs.

from datetime import datetime as _dt2

# ── 1. Per-rule waterfall & match metrics ────────────────────────────────
# Combine _rule_diagnostics with actual match counts from brd_matches
rule_match_counts = (
    brd_matches
    .groupBy("priority", "category", "description")
    .agg(
        F.count("*").alias("match_rows"),
        F.countDistinct("axis_id").alias("unique_axis"),
        F.countDistinct("fin_id").alias("unique_fin"),
        F.avg("amount_diff").alias("avg_amount_diff"),
        F.avg("amount_rel_diff").alias("avg_rel_diff"),
        F.expr("percentile_approx(amount_diff, 0.5)").alias("median_diff"),
        F.max("amount_diff").alias("max_diff"),
        F.sum(F.when(F.col("amount_diff") == 0, 1).otherwise(0)).alias("exact_matches"),
    )
    .orderBy("priority")
    .collect()
)

# Build lookup from diagnostics
diag_lookup = {d["priority"]: d for d in _rule_diagnostics}

print(f"\n{'═'*100}")
print(f"  📊 POC SUMMARY REPORT — Sequential Waterfall v5.1")
print(f"  Generated: {_dt2.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Run ID: {RUN_ID[:12]}...  Batch: {BATCH_ID}")
print(f"{'═'*100}")

print(f"\n{'─'*100}")
print(f"  1. PER-RULE WATERFALL METRICS")
print(f"{'─'*100}")
print(f"  {'P':>3s} {'Cat':>6s} {'AxisIn':>9s} {'EstRows':>11s} {'ActRows':>10s} "
      f"{'UniqAx':>9s} {'UniqFn':>9s} {'Legs/Ax':>8s} {'AvgDiff':>12s} {'Exact%':>7s}")
print(f"  {'─'*96}")

total_exact = 0
for row in rule_match_counts:
    p = row["priority"]
    d = diag_lookup.get(p, {})
    axis_in = d.get("axis_input_count", 0)
    est = d.get("est_rows", 0)
    legs_per = row["match_rows"] / max(row["unique_axis"], 1)
    exact_pct = row["exact_matches"] / max(row["match_rows"], 1) * 100
    total_exact += row["exact_matches"]
    print(f"  {p:>3d} {row['category']:>6s} {axis_in:>9,} {est:>11,} "
          f"{row['match_rows']:>10,} {row['unique_axis']:>9,} "
          f"{row['unique_fin']:>9,} {legs_per:>8.2f} "
          f"{row['avg_amount_diff']:>12,.2f} {exact_pct:>6.1f}%")

# Skipped rules
for d in sorted(_rule_diagnostics, key=lambda x: x["priority"]):
    if "SKIPPED" in d.get("status", ""):
        print(f"  {d['priority']:>3d} {d['category']:>6s} {'—':>9s} {'—':>11s} "
              f"{'SKIPPED':>10s} {'—':>9s} {'—':>9s} {'—':>8s} {'—':>12s} {'—':>7s}  "
              f"({d['status']})")

print(f"  {'─'*96}")
print(f"  TOTALS:{'':>20s} {brd_match_rows:>10,} {brd_unique_axis:>9,} "
      f"{brd_unique_fin:>9,} {brd_match_rows/max(brd_unique_axis,1):>8.2f} "
      f"{'':>12s} {total_exact/max(brd_match_rows,1)*100:>6.1f}%")

# ── 2. Waterfall pool removal summary ───────────────────────────────────
print(f"\n{'─'*100}")
print(f"  2. WATERFALL POOL REMOVAL EFFECTIVENESS")
print(f"{'─'*100}")
print(f"  Original Axis pool: {ORIGINAL_AXIS_COUNT:,}")
cumulative_matched = 0
for d in sorted(_rule_diagnostics, key=lambda x: x["priority"]):
    p = d["priority"]
    axis_in = d.get("axis_input_count", 0)
    actual = d.get("actual_rows", 0)
    removed_before = ORIGINAL_AXIS_COUNT - axis_in if axis_in > 0 else cumulative_matched
    pct_removed = removed_before / ORIGINAL_AXIS_COUNT * 100 if ORIGINAL_AXIS_COUNT > 0 else 0
    # Find how many this rule matched
    rule_match = next((r for r in rule_match_counts if r["priority"] == p), None)
    this_matched = rule_match["unique_axis"] if rule_match else 0
    cumulative_matched += this_matched
    bar_len = int(pct_removed / 2)
    bar = "█" * bar_len + "░" * (50 - bar_len)
    status = d.get("status", "")
    if "SKIPPED" in status:
        print(f"  P{p:>2d}: SKIPPED")
    else:
        print(f"  P{p:>2d}: {axis_in:>9,} axis_in | +{this_matched:>8,} matched | "
              f"{pct_removed:>5.1f}% removed {bar}")

print(f"  {'─'*96}")
print(f"  Final: {brd_unique_axis:,} unique Axis matched "
      f"({brd_unique_axis/ORIGINAL_AXIS_COUNT*100:.2f}%)")

# ── 3. Match quality summary ────────────────────────────────────────────
print(f"\n{'─'*100}")
print(f"  3. MATCH QUALITY SUMMARY")
print(f"{'─'*100}")

quality_stats = brd_matches.agg(
    F.avg("amount_diff").alias("avg_diff"),
    F.expr("percentile_approx(amount_diff, 0.5)").alias("median_diff"),
    F.expr("percentile_approx(amount_diff, 0.95)").alias("p95_diff"),
    F.max("amount_diff").alias("max_diff"),
    F.sum(F.when(F.col("amount_diff") == 0, 1).otherwise(0)).alias("exact_count"),
    F.sum(F.when(F.col("amount_rel_diff") < 0.001, 1).otherwise(0)).alias("within_01pct"),
    F.sum(F.when(F.col("amount_rel_diff") < 0.01, 1).otherwise(0)).alias("within_1pct"),
    F.count("*").alias("total"),
).first()

print(f"  Amount difference (|axis - fin|):")
print(f"    Average:             {quality_stats['avg_diff']:>15,.2f}")
print(f"    Median:              {quality_stats['median_diff']:>15,.2f}")
print(f"    P95:                 {quality_stats['p95_diff']:>15,.2f}")
print(f"    Max:                 {quality_stats['max_diff']:>15,.2f}")
print(f"  Match precision:")
exact_pct = quality_stats['exact_count'] / max(quality_stats['total'], 1) * 100
within_01 = quality_stats['within_01pct'] / max(quality_stats['total'], 1) * 100
within_1 = quality_stats['within_1pct'] / max(quality_stats['total'], 1) * 100
print(f"    Exact (diff=0):      {quality_stats['exact_count']:>10,}  ({exact_pct:.1f}%)")
print(f"    Within 0.1%:         {quality_stats['within_01pct']:>10,}  ({within_01:.1f}%)")
print(f"    Within 1.0%:         {quality_stats['within_1pct']:>10,}  ({within_1:.1f}%)")

# ── 4. P15 before/after comparison ──────────────────────────────────────
p15_diag = diag_lookup.get(15)
p15_match = next((r for r in rule_match_counts if r["priority"] == 15), None)
if p15_diag:
    print(f"\n{'─'*100}")
    print(f"  4. P15 EXPLOSION FIX — BEFORE vs AFTER")
    print(f"{'─'*100}")
    print(f"  {'Metric':<40s} {'Before (v4.x)':>15s} {'After (v5.0)':>15s} {'Reduction':>12s}")
    print(f"  {'─'*85}")

    # Before values (from production observation)
    before_rows = 101_627_117
    before_axis = ORIGINAL_AXIS_COUNT  # all axis participated
    after_axis_in = p15_diag.get("axis_input_count", 0)
    after_rows = p15_diag.get("actual_rows", 0)

    reduction_rows = before_rows - after_rows
    reduction_pct = reduction_rows / max(before_rows, 1) * 100

    print(f"  {'Axis input to P15':<40s} {before_axis:>15,} {after_axis_in:>15,} "
          f"{(before_axis-after_axis_in)/max(before_axis,1)*100:>10.1f}%")
    print(f"  {'Estimated join rows':<40s} {before_rows:>15,} {p15_diag['est_rows']:>15,} "
          f"{'—':>12s}")
    print(f"  {'Actual output rows':<40s} {before_rows:>15,} {after_rows:>15,} "
          f"{reduction_pct:>10.1f}%")
    if p15_match:
        print(f"  {'Unique Axis matched by P15':<40s} {'?':>15s} "
              f"{p15_match['unique_axis']:>15,} {'—':>12s}")
        print(f"  {'Avg amount diff':<40s} {'?':>15s} "
              f"{p15_match['avg_amount_diff']:>15,.2f} {'—':>12s}")

# ── 5. Final scorecard ──────────────────────────────────────────────────
print(f"\n{'─'*100}")
print(f"  5. FINAL SCORECARD")
print(f"{'─'*100}")
print(f"  {'Metric':<40s} {'Value':>15s}")
print(f"  {'─'*60}")
print(f"  {'Axis (in-scope)':<40s} {ORIGINAL_AXIS_COUNT:>15,}")
print(f"  {'Finstore (total)':<40s} {ORIGINAL_FINSTORE_COUNT:>15,}")
print(f"  {'BRD unique Axis matched':<40s} {brd_unique_axis:>15,}")
print(f"  {'BRD match rate':<40s} {brd_match_rate:>14.2f}%")
print(f"  {'Greedy additional matches':<40s} {total_greedy:>15,}")
print(f"  {'TOTAL unique Axis matched':<40s} {total_matched:>15,}")
print(f"  {'TOTAL match rows (incl multi-leg)':<40s} {total_match_rows:>15,}")
print(f"  {'FINAL MATCH RATE':<40s} {final_match_rate:>14.2f}%")
print(f"  {'Unmatched Axis':<40s} {ORIGINAL_AXIS_COUNT - total_matched:>15,}")
print(f"  {'Multi-leg extra rows (BRD)':<40s} {brd_match_rows - brd_unique_axis:>15,}")
print(f"  {'Exact amount matches (BRD)':<40s} {quality_stats['exact_count']:>15,}")

print(f"\n{'═'*100}")
print(f"  POC SUMMARY REPORT COMPLETE")
print(f"{'═'*100}")

---
## 🔹 Section 23: Verdict

Pass / fail assessment against the 85% match rate target.

In [ ]:
# ============================================================
# SECTION 23: VERDICT
# ============================================================
# Identical target thresholds as the Pandas reference notebook
# (85% = success, 80–84.99% = acceptable, <80% = fail).
# Uses the same final_match_rate_report variable computed in Section 22.

TARGET_RATE = 85.0
ACCEPTABLE_RATE = 80.0

print(f"\n{'='*80}")
print("VERDICT")
print(f"{'='*80}")
print(f"\n  Target           : {TARGET_RATE:.0f}%")
print(f"  Achieved         : {final_match_rate_report:.2f}%")
print(f"  BRD contribution : {brd_match_rate_report:.2f}%")
print(f"  Greedy contrib.  : {greedy_match_rate_report:.2f}%")
print(f"  Unmatched (Axis) : {unmatched_count_report:,}")
print()

if final_match_rate_report >= TARGET_RATE:
    verdict_label  = "✅  TARGET ACHIEVED"
    verdict_status = "SUCCESS"
    verdict_note   = f"Exceeded target by {final_match_rate_report - TARGET_RATE:.2f} percentage points."

elif final_match_rate_report >= ACCEPTABLE_RATE:
    verdict_label  = "⚠️   TARGET NEARLY ACHIEVED"
    verdict_status = f"ACCEPTABLE  (within {ACCEPTABLE_RATE:.0f}–{TARGET_RATE:.0f}% range)"
    verdict_note   = (
        f"Gap to target: {TARGET_RATE - final_match_rate_report:.2f} pp.  "
        f"Review unmatched_reason breakdown (Section 18) for remediation."
    )

else:
    verdict_label  = "❌  TARGET NOT ACHIEVED"
    verdict_status = "FAIL"
    verdict_note   = (
        f"Gap to target: {TARGET_RATE - final_match_rate_report:.2f} pp.  "
        f"Review unmatched_reason breakdown (Section 18) and consider "
        f"expanding greedy tolerances or adding new BRD rules."
    )

print(f"  {verdict_label}")
print(f"  Status : {verdict_status}")
print(f"  Note   : {verdict_note}")

print(f"\n{'='*80}")
print("Notebook execution complete.")
print(f"  Run ID   : {RUN_ID}")
print(f"  Batch ID : {BATCH_ID}")
print(f"{'='*80}")

---
## Section 24: Standalone Reconciliation Summary Report

Loads saved Delta tables from `OUTPUT_DIR` and prints a reconciliation
summary.  **Only requires** a Spark session and the config cell (Section 2)
to have been run.  All other pipeline state is optional — the report
auto-detects which columns and tables are available and skips sections
gracefully when data is missing.

In [ ]:
# ============================================================
# SECTION 24 — STANDALONE RECONCILIATION SUMMARY  (v5.5)
# ============================================================
# Loads Delta tables from OUTPUT_DIR and prints a full report.
# Auto-detects available columns; never crashes on missing data.
# ============================================================

from datetime import datetime as _dt
from pyspark.sql import functions as F

_W = 80

def _hdr(title, level=1):
    ch = "=" if level == 1 else "─"
    print(f"\n{ch * _W}")
    print(f"  {title}")
    print(f"{ch * _W}")

def _safe(col_name, df):
    """True if col_name exists in df."""
    return col_name in df.columns

# ── Config defaults (safe if config cell was not run) ─────────────
try:
    _out = OUTPUT_DIR
except NameError:
    _out = "/mnt/data/derivatives/matching_results"

try:
    _excl = EXCLUDE_SOPHIS_DELTA1
    _oos  = OUT_OF_SCOPE_SYSTEMS
except NameError:
    _excl = True
    _oos  = ["SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO",
             "SOPHISFX-LONDON", "DELTA1-LONDON", "DELTA1-NEWYORK"]

try:    _gtol = GREEDY_TOLERANCE_PCT
except NameError: _gtol = 0.01

try:    _stol = STRICT_TOLERANCE_PCT
except NameError: _stol = 0.001


# =================================================================
# 0. DATA LOAD
# =================================================================
_hdr("DERIVATIVES TRADE MATCHING — RECONCILIATION REPORT")
print(f"  Generated : {_dt.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Source    : {_out}\n")

_matched = spark.read.format("delta").load(f"{_out}/matched_all_base")
_unmatched_axis = spark.read.format("delta").load(f"{_out}/unmatched_axis_base")

try:
    _unmatched_fin = spark.read.format("delta").load(f"{_out}/unmatched_finstore_base")
except Exception:
    _unmatched_fin = None

_matched.cache()
_unmatched_axis.cache()
if _unmatched_fin:
    _unmatched_fin.cache()

_cols = set(_matched.columns)
print(f"  matched_all_base   : {len(_cols)} cols")
print(f"  unmatched_axis_base: loaded")
print(f"  unmatched_finstore : {'loaded' if _unmatched_fin else 'not available'}")

# ── Core counts ──────────────────────────────────────────────────
_total_rows      = _matched.count()
_unique_axis     = _matched.select("axis_id").distinct().count()
_unique_fin      = _matched.select("fin_id").distinct().count()
_unmatched_count = _unmatched_axis.count()
_scope           = _unique_axis + _unmatched_count

_brd   = _matched.filter(F.col("MatchLayer") == "BRD")
_grdy  = _matched.filter(F.col("MatchLayer") == "GREEDY")
_brd_rows      = _brd.count()
_brd_ax        = _brd.select("axis_id").distinct().count()
_brd_fin       = _brd.select("fin_id").distinct().count()
_grdy_rows     = _grdy.count()
_grdy_ax       = _grdy.select("axis_id").distinct().count()

_brd_pct  = _brd_ax  / max(_scope, 1) * 100
_grdy_pct = _grdy_ax / max(_scope, 1) * 100
_rate     = _unique_axis / max(_scope, 1) * 100


# =================================================================
# 1. RUN METADATA  (optional — only if columns exist)
# =================================================================
_meta_cols = ["run_id", "batch_id", "rule_version", "match_timestamp",
              "verification_status", "greedy_status"]
_available_meta = [c for c in _meta_cols if c in _cols]

_hdr("1. RUN METADATA")
if _available_meta:
    _row = _matched.select(*_available_meta).first()
    for c in _available_meta:
        print(f"    {c:<24s}: {_row[c]}")
else:
    print("  Metadata columns not present in saved data (pipeline may not")
    print("  have reached the save-with-metadata step). Skipping.")


# =================================================================
# 2. KEY PERFORMANCE INDICATORS
# =================================================================
_hdr("2. KEY PERFORMANCE INDICATORS")
print(f"""
  CONFIRMED  = BRD deterministic rules (high confidence)
  INDICATIVE = Greedy probabilistic    (review borderline)
  UNMATCHED  = No Finstore counterpart

  {'Category':<44s} {'Trades':>10s} {'%':>8s}
  {'─'*64}
  {'Axis in scope':<44s} {_scope:>10,} {'100.00':>7s}%
  {'CONFIRMED  (BRD)':<44s} {_brd_ax:>10,} {_brd_pct:>7.2f}%
  {'INDICATIVE (Greedy)':<44s} {_grdy_ax:>10,} {_grdy_pct:>7.2f}%
  {'UNMATCHED':<44s} {_unmatched_count:>10,} {100-_rate:>7.2f}%
  {'─'*64}
  {'TOTAL MATCHED':<44s} {_unique_axis:>10,} {_rate:>7.2f}%
""")
_extra = _total_rows - _unique_axis
if _extra > 0:
    print(f"  ({_extra:,} extra rows from multi-leg instruments)")


# =================================================================
# 3. EXCLUDED TRADES
# =================================================================
_hdr("3. EXCLUDED TRADES")
print(f"  EXCLUDE_SOPHIS_DELTA1 = {_excl}")
if _excl:
    print(f"  Systems removed: {', '.join(_oos)}")
    print(f"  Reason: trade IDs from these systems don't exist in Finstore.")
else:
    print(f"  All systems in scope.")


# =================================================================
# 4. BRD RULES (per-rule breakdown)
# =================================================================
_hdr("4. BRD DETERMINISTIC RULES")

_rs = (
    _brd
    .groupBy("priority", "category", "description")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("axis_id").alias("ax"),
        F.countDistinct("fin_id").alias("fn"),
        F.avg("amount_diff").alias("avg"),
        F.expr("percentile_approx(amount_diff, 0.5)").alias("med"),
        F.sum(F.when(F.col("amount_diff") == 0, 1).otherwise(0)).alias("exact"),
    )
    .orderBy("priority")
    .collect()
)

_act = {r["priority"] for r in _rs}
_sop = {1, 2, 3}

print(f"  {'P':>2s} {'Cat':>6s} {'Stat':<7s} {'Rows':>8s} {'Axis':>7s} {'Fin':>7s}"
      f" {'Avg':>10s} {'Med':>10s} {'Exact%':>7s}")
print(f"  {'─'*70}")

_tex = 0
for p in range(1, 16):
    r = next((x for x in _rs if x["priority"] == p), None)
    if r:
        ep = r["exact"] / max(r["rows"], 1) * 100
        _tex += r["exact"]
        print(f"  {p:>2d} {r['category']:>6s} {'ACTIVE':<7s} {r['rows']:>8,} "
              f"{r['ax']:>7,} {r['fn']:>7,} {r['avg']:>10,.2f} {r['med']:>10,.2f} {ep:>6.1f}%")
    else:
        cat = "SOPHIS" if p in _sop else ("ETD" if p >= 14 else "OTC")
        st  = "SKIP" if (p in _sop and _excl) else "ZERO"
        print(f"  {p:>2d} {cat:>6s} {st:<7s} {'--':>8s} {'--':>7s} {'--':>7s}"
              f" {'--':>10s} {'--':>10s} {'--':>7s}")

print(f"  {'─'*70}")
print(f"  {'TOT':>9s} {_brd_rows:>8,} {_brd_ax:>7,} {_brd_fin:>7,}"
      f" {'':>10s} {'':>10s} {_tex/max(_brd_rows,1)*100:>6.1f}%")


# =================================================================
# 5. GREEDY MATCHING
# =================================================================
_hdr("5. GREEDY PROBABILISTIC MATCHING")
print(f"  S1 tolerance: {_gtol*100:.1f}%   S2 tolerance: {_stol*100:.2f}%\n")

if _grdy_rows > 0:
    _gd = (
        _grdy.groupBy("priority", "description")
        .agg(F.count("*").alias("n"), F.avg("amount_diff").alias("avg"),
             F.max("amount_diff").alias("mx"))
        .orderBy("priority").collect()
    )
    print(f"  {'P':>2s} {'Description':<45s} {'N':>8s} {'AvgDiff':>12s} {'MaxDiff':>12s}")
    print(f"  {'─'*82}")
    for r in _gd:
        print(f"  {r['priority']:>2d} {r['description']:<45s} {r['n']:>8,}"
              f" {r['avg']:>12,.2f} {r['mx']:>12,.2f}")
    print(f"  {'─'*82}")
    print(f"  {'Total':>47s} {_grdy_rows:>8,}")
else:
    print(f"  No greedy matches produced.")


# =================================================================
# 6. 1:1 TRACEABILITY (v5.5 — if columns present)
# =================================================================
_hdr("6. 1:1 TRACEABILITY")

if "is_best_match" in _cols:
    _bc = _matched.filter(F.col("is_best_match")).select("axis_id").distinct().count()
    _1c = _matched.filter(F.col("match_type") == "1:1").select("axis_id").distinct().count()
    _mc = _matched.filter(F.col("match_type") == "multi_leg").select("axis_id").distinct().count()

    print(f"  {'Metric':<40s} {'Count':>10s} {'%':>8s}")
    print(f"  {'─'*60}")
    print(f"  {'is_best_match = True':<40s} {_bc:>10,} {_bc/max(_unique_axis,1)*100:>7.1f}%")
    print(f"  {'match_type = 1:1':<40s} {_1c:>10,} {_1c/max(_unique_axis,1)*100:>7.1f}%")
    print(f"  {'match_type = multi_leg':<40s} {_mc:>10,} {_mc/max(_unique_axis,1)*100:>7.1f}%")
    print(f"\n  Tip: filter is_best_match = True for one-row-per-axis output.")
else:
    print(f"  Columns not present (data from pre-v5.5 run). Skipping.")


# =================================================================
# 7. MATCH QUALITY
# =================================================================
_hdr("7. MATCH QUALITY")

_q = _matched.agg(
    F.avg("amount_diff").alias("avg"),
    F.expr("percentile_approx(amount_diff, 0.5)").alias("med"),
    F.expr("percentile_approx(amount_diff, 0.95)").alias("p95"),
    F.max("amount_diff").alias("mx"),
    F.sum(F.when(F.col("amount_diff") == 0, 1).otherwise(0)).alias("exact"),
    F.sum(F.when(F.col("amount_rel_diff") < 0.01, 1).otherwise(0)).alias("w1"),
    F.count("*").alias("n"),
).first()

print(f"  Average diff : {_q['avg']:>14,.2f}")
print(f"  Median  diff : {_q['med']:>14,.2f}")
print(f"  P95    diff  : {_q['p95']:>14,.2f}")
print(f"  Max    diff  : {_q['mx']:>14,.2f}")
print(f"\n  Exact matches        : {_q['exact']:>10,}  ({_q['exact']/max(_q['n'],1)*100:.1f}%)")
print(f"  Within 1% relative   : {_q['w1']:>10,}  ({_q['w1']/max(_q['n'],1)*100:.1f}%)")


# =================================================================
# 8. MATCHES BY SOURCE SYSTEM  (needs enriched table)
# =================================================================
_hdr("8. MATCHES BY SOURCE SYSTEM")

try:
    _enr = spark.read.format("delta").load(f"{_out}/matched_all_enriched")
    _sc = "SourceSystemName_Axis" if "SourceSystemName_Axis" in _enr.columns else "SourceSystemName"

    _sb = (
        _enr.filter(F.col("MatchLayer") == "BRD")
        .groupBy(_sc).agg(F.countDistinct("axis_id").alias("ax"))
        .orderBy(F.desc("ax")).collect()
    )
    print(f"\n  BRD (confirmed):")
    print(f"  {'System':<30s} {'Axis':>10s}")
    print(f"  {'─'*42}")
    for r in _sb:
        print(f"  {str(r[_sc]):<30s} {r['ax']:>10,}")

    if _grdy_rows > 0:
        _sg = (
            _enr.filter(F.col("MatchLayer") == "GREEDY")
            .groupBy(_sc).agg(F.countDistinct("axis_id").alias("ax"))
            .orderBy(F.desc("ax")).collect()
        )
        print(f"\n  Greedy (indicative):")
        print(f"  {'System':<30s} {'Axis':>10s}")
        print(f"  {'─'*42}")
        for r in _sg:
            print(f"  {str(r[_sc]):<30s} {r['ax']:>10,}")

    _enr.unpersist()
except Exception as _e:
    print(f"  Enriched table not available ({_e}). Skipping.")


# =================================================================
# 9. UNMATCHED BY SOURCE SYSTEM
# =================================================================
_hdr("9. UNMATCHED BY SOURCE SYSTEM")

_um_col = None
if "SourceSystemName" in _unmatched_axis.columns:
    _um_col = "SourceSystemName"

if _um_col is None:
    try:
        _ue = spark.read.format("delta").load(f"{_out}/unmatched_axis_enriched")
        _um_col_e = "SourceSystemName_Axis" if "SourceSystemName_Axis" in _ue.columns else "SourceSystemName"
        _um_sys = _ue.groupBy(_um_col_e).count().orderBy(F.desc("count")).collect()
        print(f"  (from unmatched_axis_enriched)")
        print(f"  {'System':<30s} {'Count':>10s}")
        print(f"  {'─'*42}")
        for r in _um_sys:
            print(f"  {str(r[_um_col_e]):<30s} {r['count']:>10,}")
    except Exception:
        print(f"  SourceSystemName not available. Skipping.")
else:
    _um_sys = (
        _unmatched_axis.groupBy(_um_col).count()
        .orderBy(F.desc("count")).collect()
    )
    print(f"  {'System':<30s} {'Count':>10s} {'%':>8s}")
    print(f"  {'─'*50}")
    for r in _um_sys:
        pct = r["count"] / max(_unmatched_count, 1) * 100
        flag = "  <-- investigate" if pct > 25 else ""
        print(f"  {str(r[_um_col]):<30s} {r['count']:>10,} {pct:>7.1f}%{flag}")


# =================================================================
# 10. DATA QUALITY
# =================================================================
_hdr("10. DATA QUALITY CHECKS")

_checks = [
    ("null axis_id",     F.col("axis_id").isNull()),
    ("null fin_id",      F.col("fin_id").isNull()),
    ("null description", F.col("description").isNull()),
]
# Only check run_id if it exists
if "run_id" in _cols:
    _checks.append(("null run_id", F.col("run_id").isNull()))

_aggs = [F.sum(F.when(e, 1).otherwise(0)).alias(d) for d, e in _checks]
_dq = _matched.agg(*_aggs).collect()[0]

_flags = 0
print(f"  {'Check':<24s} {'Issues':>10s} {'Status':>8s}")
print(f"  {'─'*44}")
for desc, _ in _checks:
    v = int(_dq[desc])
    st = "OK" if v == 0 else "FLAG"
    if v > 0: _flags += 1
    print(f"  {desc:<24s} {v:>10,} {st:>8s}")

# Duplicate pairs
_pd = _matched.select("axis_id", "fin_id").distinct().count()
_dup = _total_rows - _pd
st = "OK" if _dup == 0 else "FLAG"
if _dup > 0: _flags += 1
print(f"  {'duplicate (ax,fin) pair':<24s} {_dup:>10,} {st:>8s}")

print(f"\n  {_flags} issue(s) flagged." if _flags else "\n  All checks passed.")


# =================================================================
# 11. OUTPUT FILE REFERENCE
# =================================================================
_hdr("11. OUTPUT FILES")
print(f"""
  Delta tables under: {_out}

  {'Table':<32s} {'Schema':<8s} {'Content'}
  {'─'*70}
  {'matched_all_base':<32s} {'Narrow':<8s} All match pairs (BRD + Greedy)
  {'matched_all_enriched':<32s} {'Wide':<8s}   + all source columns
  {'matched_brd_layer':<32s} {'Narrow':<8s} BRD pairs only
  {'matched_greedy_layer':<32s} {'Narrow':<8s} Greedy pairs only
  {'unmatched_axis_base':<32s} {'Narrow':<8s} Unmatched Axis trades
  {'unmatched_finstore_base':<32s} {'Narrow':<8s} Unmatched Finstore records

  For 1-row-per-Axis: SELECT * FROM matched_all_base WHERE is_best_match = True
""")


# =================================================================
# 12. VERDICT
# =================================================================
_hdr("12. VERDICT")
_TGT = 85.0
print(f"  Target: {_TGT:.0f}%   Achieved: {_rate:.2f}%")
print(f"  BRD: {_brd_pct:.2f}%  |  Greedy: {_grdy_pct:.2f}%  |  Unmatched: {_unmatched_count:,}")

if _rate >= _TGT:
    print(f"\n  ✅ TARGET ACHIEVED (+{_rate - _TGT:.2f} pp)")
elif _rate >= 80:
    print(f"\n  ⚠️  NEAR TARGET (gap: {_TGT - _rate:.2f} pp)")
else:
    print(f"\n  ❌ BELOW TARGET (gap: {_TGT - _rate:.2f} pp)")

_report_footer = f"\n{'=' * _W}\n  END OF REPORT — {_dt.now().strftime('%Y-%m-%d %H:%M:%S')}\n{'=' * _W}\n"
print(_report_footer)

# ── Save report as TXT ───────────────────────────────────────────
import io as _io, sys as _sys

_txt_path = f"{_out}/reconciliation_report_{_dt.now().strftime('%Y%m%d_%H%M%S')}.txt"

# Capture all printed output into a string buffer, then write once
_buf = _io.StringIO()
_old_stdout = _sys.stdout
_sys.stdout = _buf

# Re-run the lightweight summary block (counts already computed)
print(f"DERIVATIVES TRADE MATCHING — RECONCILIATION REPORT")
print(f"Generated : {_dt.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Source    : {_out}\n")
print(f"Match rate: {_rate:.2f}%  (BRD {_brd_pct:.2f}%  Greedy {_grdy_pct:.2f}%  Unmatched {_unmatched_count:,})")
print(f"Scope     : {_scope:,} Axis trades in scope")
print(f"Matched   : {_unique_axis:,}  ({_brd_ax:,} confirmed + {_grdy_ax:,} indicative)")
print(f"Unmatched : {_unmatched_count:,}")
if _rate >= 85:
    print(f"Verdict   : TARGET ACHIEVED (+{_rate - 85:.2f} pp)")
elif _rate >= 80:
    print(f"Verdict   : NEAR TARGET (gap {85 - _rate:.2f} pp)")
else:
    print(f"Verdict   : BELOW TARGET (gap {85 - _rate:.2f} pp)")

_sys.stdout = _old_stdout
_report_txt = _buf.getvalue()

try:
    dbutils.fs.put(_txt_path, _report_txt, overwrite=True)
    print(f"  Report saved (DBFS): {_txt_path}")
except NameError:
    # Local / non-Databricks environment
    with open(_txt_path.replace("dbfs:", ""), "w") as _f:
        _f.write(_report_txt)
    print(f"  Report saved (local): {_txt_path}")

# ── Cleanup ──────────────────────────────────────────────────────
_matched.unpersist()
_unmatched_axis.unpersist()
if _unmatched_fin:
    _unmatched_fin.unpersist()